# ALTO v1: Context-aware evaluator with adaptive latent sampling under a fixed generation budget

V1 tập trung vào context-aware evaluator và adaptive latent sampling dưới ngân sách tối đa cố định 16 generation/proposal. Đơn vị thí nghiệm là `specification × background × spatial proposal × latent seed`; schema vẫn hỗ trợ nhiều proposal trên cùng background. Evaluator tách output validity, subject validity, constraint satisfaction và continuous failure profile để điều khiển allocation.

Luồng chính: **ADE20K background → SpatialProposal selection → CandidateState → SDXL round 4 → evaluator → allocation round 4 → evaluator → allocation round 8 → final ranking và analysis**.


## Cấu trúc Drive

```text
/content/drive/MyDrive/Colab Notebooks/alto/v1/
├── Input/
│   ├── backgrounds/
│   ├── masks/
│   ├── semantic_masks/
│   ├── dataset_manifest.csv
│   └── proposal_manifest.csv
└── Output/
    ├── images/
    ├── candidate_previews/      # tùy chọn
    ├── contact_sheets/          # tùy chọn
    ├── candidate_manifest.csv
    ├── proposal_search_audit.csv
    ├── candidate_audit.csv
    ├── constraint_results.csv
    ├── failure_profiles.csv
    ├── adaptive_budget_decisions.csv
    ├── adaptive_vs_oracle.csv
    ├── adaptive_vs_oracle_summary.csv
    ├── runs.csv
    ├── ranking.csv
    ├── summary_by_context.csv
    ├── summary_by_failure.csv
    ├── summary_by_proposal.csv
    ├── variance_decomposition.csv
    ├── proposal_feasibility_correlations.csv
    ├── best_of_n_by_proposal.csv
    ├── best_of_n_summary.csv
    └── experiment_manifest.json
```

Manual review được thực hiện riêng trong `alto_v1_manual_validation.ipynb`.


## 1. Mount Drive và cài dependencies

Colab đã có sẵn PyTorch, NumPy, pandas và SciPy; cell cài đặt chỉ bổ sung các thư viện model/dataset còn thiếu.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%pip install -q -U datasets diffusers transformers accelerate safetensors timm

In [ ]:
import gc
import hashlib
import heapq
import json
import math
import time
from dataclasses import dataclass, field
from pathlib import Path
from datetime import datetime
from typing import Literal

import numpy as np
import pandas as pd
import torch

from PIL import Image, ImageChops, ImageDraw, ImageFilter
from IPython.display import display
from scipy import ndimage, stats
from datasets import load_dataset
from diffusers import AutoPipelineForInpainting
from transformers import (
    AutoImageProcessor,
    Mask2FormerForUniversalSegmentation,
    VitPoseForPoseEstimation,
)

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Not enabled")



## 2. Cấu hình thí nghiệm

Các ngưỡng chọn dữ liệu, kích thước mask, tham số sinh ảnh và model chấm điểm được tập trung trong `CONFIG`. Sau khi thay đổi rule chọn ảnh hoặc tạo mask, chạy một lần với `REBUILD_DATASET=True`; sau đó có thể đặt lại `False` để dùng cache trên Drive.

In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/alto/v1")
INPUT_DIR = PROJECT_DIR / "Input"
OUTPUT_DIR = PROJECT_DIR / "Output"

BACKGROUND_DIR = INPUT_DIR / "backgrounds"
MASK_DIR = INPUT_DIR / "masks"
SEMANTIC_DIR = INPUT_DIR / "semantic_masks"
DATASET_MANIFEST_PATH = INPUT_DIR / "dataset_manifest.csv"
PROPOSAL_MANIFEST_PATH = INPUT_DIR / "proposal_manifest.csv"
CANDIDATE_MANIFEST_PATH = OUTPUT_DIR / "candidate_manifest.csv"
PROPOSAL_SEARCH_AUDIT_PATH = OUTPUT_DIR / "proposal_search_audit.csv"
CANDIDATE_AUDIT_PATH = OUTPUT_DIR / "candidate_audit.csv"
CONSTRAINT_RESULTS_PATH = OUTPUT_DIR / "constraint_results.csv"
FAILURE_PROFILES_PATH = OUTPUT_DIR / "failure_profiles.csv"
BUDGET_DECISIONS_PATH = OUTPUT_DIR / "adaptive_budget_decisions.csv"
ORACLE_COMPARISON_PATH = OUTPUT_DIR / "adaptive_vs_oracle.csv"
ORACLE_SUMMARY_PATH = OUTPUT_DIR / "adaptive_vs_oracle_summary.csv"

# Bật True sau khi đổi rule chọn ảnh/mask; đặt False sau lần rebuild thành công.
REBUILD_DATASET = True
RUN_GENERATION = True
RUN_EVALUATION = True
# Preview/contact sheet không ảnh hưởng thí nghiệm; chỉ bật khi cần kiểm tra trực quan.
CREATE_PREVIEWS = False

for path in [INPUT_DIR, OUTPUT_DIR, BACKGROUND_DIR, MASK_DIR, SEMANTIC_DIR, OUTPUT_DIR / "images"]:
    path.mkdir(parents=True, exist_ok=True)
if CREATE_PREVIEWS:
    for path in [OUTPUT_DIR / "candidate_previews", OUTPUT_DIR / "contact_sheets"]:
        path.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "experiment_name": "alto_v1_context_evaluator_adaptive_sampling",
    # Dataset streaming và candidate search.
    "dataset_id": "zhoubolei/scene_parse_150",
    "dataset_split": "validation",
    "dataset_seed": 42,
    "max_stream_examples": 10000,
    "shuffle_buffer_size": 500,
    "top_k_candidates_per_spec": 12,
    "min_candidates_before_early_stop": 12,
    "backgrounds_per_spec": 4,

    # Ngưỡng hình học khi chọn background và vị trí.
    "strict_min_foot_ground_ratio": 0.25,
    "strict_max_obstacle_ratio": 0.34,
    "strict_max_relative_distance": 0.40,
    "relaxed_min_foot_ground_ratio": 0.05,
    "relaxed_max_obstacle_ratio": 0.62,
    "relaxed_max_relative_distance": 0.55,

    # Kích thước và mức bảo vệ target mask.
    "standing_mask_width_ratio": 0.24,
    "standing_mask_height_ratio": 0.72,
    "relative_mask_width_ratio": 0.24,
    "relative_mask_height_ratio": 0.72,
    "sitting_mask_width_ratio": 0.30,
    "sitting_mask_height_ratio": 0.62,
    "shadow_extension_ratio": 0.04,
    "mask_corner_radius_ratio": 0.22,
    "reference_protection_dilation": 4,

    # Adaptive latent sampling và Best-of-N reporting.
    "max_seeds_per_proposal": 16,
    "seed_start": 1000,
    "best_of_budgets": [1, 4, 8, 16],
    "adaptive_budget_enabled": True,
    # full_fixed sinh đủ pool để đánh giá policy offline; online_adaptive tiết kiệm compute.
    "generation_protocol": "full_fixed",
    "adaptive_round_seed_counts": [4, 4, 8],
    "adaptive_near_hard_margin": 0.05,
    # Chỉ dùng cho diagnostic; không được dùng làm hard early-stop.
    "adaptive_severe_context_damage": 0.75,
    # Chỉ dừng no-person sau khi đã quan sát ít nhất 8 seed.
    "adaptive_min_no_person_seeds": 8,

    # Kích thước là giới hạn tối đa; resize vẫn giữ nguyên tỷ lệ ảnh.
    "model_id": "diffusers/stable-diffusion-xl-1.0-inpainting-0.1",
    "width": 1024,
    "height": 1024,
    "num_inference_steps": 40,
    "guidance_scale": 8.0,
    "strength": 0.99,
    "mask_blur_radius": 16,

    # Automatic scorer.
    "instance_segmentation_model_id": "facebook/mask2former-swin-small-coco-instance",
    "pose_model_id": "usyd-community/vitpose-base-simple",
    "person_detection_threshold": 0.35,
    "keypoint_threshold": 0.30,
    "hard_min_placement": 0.35,
    "hard_min_scale": 0.20,
    "hard_min_relation": 0.25,

    # Điều khiển budget allocation; không tác động hard gate hoặc ranking score.
    "promising_soft_threshold": 0.55,
    "accept_soft_threshold": 0.70,
}

if CONFIG["min_candidates_before_early_stop"] > CONFIG["top_k_candidates_per_spec"]:
    raise ValueError("min_candidates_before_early_stop không được lớn hơn top_k_candidates_per_spec.")
if max(CONFIG["best_of_budgets"]) > CONFIG["max_seeds_per_proposal"]:
    raise ValueError("Best-of-N budget không được lớn hơn max_seeds_per_proposal.")
if sum(CONFIG["adaptive_round_seed_counts"]) != CONFIG["max_seeds_per_proposal"]:
    raise ValueError("Tổng adaptive round budget phải bằng max_seeds_per_proposal.")
for key in [
    "promising_soft_threshold", "accept_soft_threshold",
    "adaptive_near_hard_margin", "adaptive_severe_context_damage",
]:
    validate_value = CONFIG[key]
    if not 0.0 <= validate_value <= 1.0:
        raise ValueError(f"{key} phải nằm trong [0, 1].")
min_no_person_seeds = CONFIG["adaptive_min_no_person_seeds"]
if not isinstance(min_no_person_seeds, int) or isinstance(min_no_person_seeds, bool):
    raise TypeError("adaptive_min_no_person_seeds phải là số nguyên.")
if not 1 <= min_no_person_seeds <= CONFIG["max_seeds_per_proposal"]:
    raise ValueError(
        "adaptive_min_no_person_seeds phải nằm trong [1, max_seeds_per_proposal]."
    )
if CONFIG["promising_soft_threshold"] > CONFIG["accept_soft_threshold"]:
    raise ValueError("promising_soft_threshold không được lớn hơn accept_soft_threshold.")
if CONFIG["generation_protocol"] not in {"full_fixed", "online_adaptive"}:
    raise ValueError("generation_protocol phải là full_fixed hoặc online_adaptive.")

print(json.dumps(CONFIG, indent=2))

### Quyết định phạm vi v1

V1 tập trung vào context-aware constraint evaluator và adaptive latent sampling với ngân sách tối đa cố định. Multi-background variance, proposal correlation và Best-of-N được giữ làm analysis phụ. Các phần chưa đủ ổn định được tách khỏi baseline chính:

- Bỏ CLIP crop-quality và prompt score; anatomy/photorealism được kiểm tra bằng manual review riêng.
- Adaptive budget dùng policy audit được 4 → 4 → 8; chưa thực hiện latent perturbation hoặc parent–child refinement.
- Manual validation nằm trong `alto_v1_manual_validation.ipynb` và chỉ đọc output của notebook này.
- Candidate preview, dataset cache preview và contact sheet đều được điều khiển bằng `CREATE_PREVIEWS`.

Thiết kế mặc định có 24 proposal, tối thiểu 96 generation ở round 0 và tối đa 384 generation nếu mọi proposal dùng đủ 16 seed. Có thể chạy lại evaluation mà không tải diffusion model bằng `RUN_GENERATION=False`.


## 3. Nguồn ADE20K Parquet

Đọc trực tiếp validation Parquet ở chế độ streaming để không phải tải và giải nén toàn bộ SceneParse150 lên Drive.

In [ ]:
ADE20K_VALIDATION_PARQUET = (
    "https://huggingface.co/datasets/"
    "zhoubolei/scene_parse_150/resolve/"
    "refs%2Fconvert%2Fparquet/"
    "scene_parsing/validation/0000.parquet"
)

print("ADE20K validation source:")
print(ADE20K_VALIDATION_PARQUET)

## 4. Các lớp ADE20K cần dùng

SceneParse150 sử dụng dense semantic annotation. Notebook chỉ truy vấn một tập lớp nhỏ, không phân tích toàn bộ ontology.

In [ ]:
ADE = {
    "wall": 1, "sky": 3, "floor": 4, "road": 7, "grass": 10,
    "sidewalk": 12, "person": 13, "earth": 14, "table": 16,
    "chair": 20, "car": 21, "sofa": 24, "armchair": 31,
    "seat": 32, "path": 53, "bench": 70,
}

INDOOR_GROUND = {ADE["floor"]}

OUTDOOR_GROUND = {
    ADE["road"], ADE["grass"], ADE["sidewalk"],
    ADE["earth"], ADE["path"],
}

ALL_GROUND = INDOOR_GROUND | OUTDOOR_GROUND
PERSON_LABEL = ADE["person"]

## 5. Định nghĩa context specifications

`ContextSpecification` là biểu diễn chuẩn của một context. Mỗi specification gom subject, action, reference object và toàn bộ hard/soft constraints vào cùng một schema. Evaluator đọc `pose_mode`, `contact_mode` và `relation mode` từ `ConstraintSpec.params`; proposal search đọc spatial rules từ metadata. `context_type` chỉ còn là nhãn nhóm để tương thích báo cáo cũ, không điều khiển scoring hoặc geometry.

Candidate search, generation và evaluator dùng trực tiếp `ContextSpecification`; notebook không duy trì dictionary mirror của specification.


In [ ]:
@dataclass
class ConstraintSpec:
    name: str
    evaluator: str
    importance: Literal["hard", "soft"]
    threshold: float
    weight: float
    params: dict = field(default_factory=dict)

    def __post_init__(self):
        if self.importance not in {"hard", "soft"}:
            raise ValueError(f"importance không hợp lệ: {self.importance}")
        if not 0.0 <= self.threshold <= 1.0:
            raise ValueError(f"threshold của {self.name} phải nằm trong [0, 1]")
        if self.weight < 0.0:
            raise ValueError(f"weight của {self.name} không được âm")


@dataclass
class ContextSpecification:
    spec_id: str
    context_type: str
    prompt: str
    negative_prompt: str
    subject: dict
    action: dict | None
    reference: dict | None
    constraints: list[ConstraintSpec]
    metadata: dict = field(default_factory=dict)

    def __post_init__(self):
        if not any(constraint.weight > 0.0 for constraint in self.constraints):
            raise ValueError(f"{self.spec_id} phải có ít nhất một constraint có weight > 0")


NEGATIVE_PROMPT = (
    "empty scene, cropped body, headless, faceless, missing limbs, extra limbs, "
    "fused limbs, multiple people, floating body, malformed anatomy, malformed face, "
    "malformed hands, malformed feet, wrong scale, wrong perspective, ghost silhouette, "
    "rectangular seam, distorted furniture, blurry, low quality"
)


# Giữ nguyên weights hiện tại nhưng tách chúng khỏi context_type.
FREE_STANDING_WEIGHTS = {
    "placement": .25, "scale": .25, "pose_contact": .30,
    "context_preservation": .20,
}
RELATIVE_STANDING_WEIGHTS = {
    "placement": .20, "scale": .15, "pose_contact": .20,
    "relation": .15, "reference_preservation": .15,
    "context_preservation": .15,
}
SITTING_INTERACTION_WEIGHTS = {
    "placement": .10, "scale": .10, "pose_contact": .30,
    "relation": .20, "reference_preservation": .15,
    "context_preservation": .15,
}

FREE_SPATIAL_RULES = {
    "proposal_strategy": "free_space",
    "ground_contact": True, "sky_clearance": True,
    "reference_relation": None, "allow_reference_occlusion": False,
}
RELATIVE_SPATIAL_RULES = {
    "proposal_strategy": "object_relative",
    "ground_contact": True, "sky_clearance": True,
    "reference_relation": "clearance", "allow_reference_occlusion": False,
}
INTERACTION_SPATIAL_RULES = {
    "proposal_strategy": "object_interaction",
    "ground_contact": False, "sky_clearance": False,
    "reference_relation": "overlap", "allow_reference_occlusion": True,
}

CONSTRAINT_EVALUATORS = {
    "output_valid": "generation_status",
    "single_target_person": "person_detection",
    "placement": "placement_score",
    "scale": "scale_score",
    "pose_contact": "pose_score",
    "relation": "relation_score",
    "context_preservation": "context_preservation_score",
    "reference_preservation": "reference_preservation_score",
    "interaction": "contact_score",
    "manual_photorealism": "manual_photorealism",
}


def make_constraints(ranking_weight_map, pose_mode, relation_mode=None):
    """Tạo constraints từ evaluator parameters, không suy luận bằng context_type."""
    constraints = [
        ConstraintSpec("output_valid", CONSTRAINT_EVALUATORS["output_valid"], "hard", 1.0, 0.0),
        ConstraintSpec(
            "single_target_person", CONSTRAINT_EVALUATORS["single_target_person"],
            "hard", 1.0, 0.0,
        ),
    ]
    hard_thresholds = {
        "placement": CONFIG["hard_min_placement"],
        "scale": CONFIG["hard_min_scale"],
    }
    if relation_mode is not None:
        hard_thresholds["relation"] = CONFIG["hard_min_relation"]

    constraint_names = dict.fromkeys([*hard_thresholds, *ranking_weight_map])
    for name in constraint_names:
        weight = ranking_weight_map.get(name, 0.0)
        params = {}
        if name == "pose_contact":
            params = {"pose_mode": pose_mode, "contact_mode": (
                "reference" if pose_mode == "sitting" else "ground"
            )}
        elif name == "relation":
            params = {"mode": relation_mode}
        constraints.append(ConstraintSpec(
            name, CONSTRAINT_EVALUATORS[name],
            "hard" if name in hard_thresholds else "soft",
            hard_thresholds.get(name, 0.0), weight, params,
        ))
    if pose_mode == "sitting":
        constraints.append(ConstraintSpec(
            "interaction", CONSTRAINT_EVALUATORS["interaction"],
            "soft", 0.0, 0.0, {"diagnostic_only": True},
        ))
    constraints.append(ConstraintSpec(
        "manual_photorealism", CONSTRAINT_EVALUATORS["manual_photorealism"],
        "soft", 0.0, 0.0, {"manual_only": True},
    ))
    return constraints


### 5.1. Context specifications

Sáu specification hiện tại dùng chung schema nhưng khai báo prompt, reference, evaluator parameters và spatial rules độc lập.


In [ ]:
CONTEXT_SPECIFICATIONS = [
    ContextSpecification(
        spec_id="spec_001",
        context_type="free_placement",
        prompt=(
            "A photorealistic adult woman standing naturally in the empty space on the "
            "left side indoors. Complete body visible: head, torso, two arms, two legs, and "
            "both feet. Feet resting naturally on the floor. Natural anatomy and scene-matched "
            "scale, perspective, lighting, and shadows."
        ),
        negative_prompt=NEGATIVE_PROMPT,
        subject={"type": "adult_person", "gender": "woman"},
        action={"type": "standing", "relation": "left_side"},
        reference=None,
        constraints=make_constraints(FREE_STANDING_WEIGHTS, "standing"),
        metadata={
            "context_type": "free_placement", "ground_labels": INDOOR_GROUND,
            **FREE_SPATIAL_RULES,
        },
    ),
    ContextSpecification(
        spec_id="spec_002",
        context_type="free_placement",
        prompt=(
            "A photorealistic adult man standing naturally in the exact center foreground "
            "outdoors. Complete body visible: head, torso, two arms, two legs, and both feet. "
            "Feet resting naturally on the ground. Natural anatomy and scene-matched scale, "
            "perspective, lighting, and shadows."
        ),
        negative_prompt=NEGATIVE_PROMPT,
        subject={"type": "adult_person", "gender": "man"},
        action={"type": "standing", "relation": "center_foreground"},
        reference=None,
        constraints=make_constraints(FREE_STANDING_WEIGHTS, "standing"),
        metadata={
            "context_type": "free_placement", "ground_labels": OUTDOOR_GROUND,
            **FREE_SPATIAL_RULES,
        },
    ),
    ContextSpecification(
        spec_id="spec_003",
        context_type="object_relative",
        prompt=(
            "A photorealistic adult woman standing naturally beside the visible table. "
            "Complete body visible: head, torso, two arms, two legs, and both feet. Feet "
            "resting naturally on the floor. Natural anatomy and scene-matched scale, "
            "perspective, lighting, and shadows."
        ),
        negative_prompt=NEGATIVE_PROMPT,
        subject={"type": "adult_person", "gender": "woman"},
        action={"type": "standing", "relation": "beside"},
        reference={
            "object": "table", "object_labels": {ADE["table"]},
            "scale_range": (1.2, 2.8),
        },
        constraints=make_constraints(
            RELATIVE_STANDING_WEIGHTS, "standing", "proximity"
        ),
        metadata={
            "context_type": "object_relative", "ground_labels": ALL_GROUND,
            **RELATIVE_SPATIAL_RULES,
        },
    ),
    ContextSpecification(
        spec_id="spec_004",
        context_type="object_relative",
        prompt=(
            "A photorealistic adult man standing naturally beside the visible car. Complete "
            "body visible: head, torso, two arms, two legs, and both feet. Feet resting "
            "naturally on the road. Natural anatomy and scene-matched scale, perspective, "
            "lighting, and shadows."
        ),
        negative_prompt=NEGATIVE_PROMPT,
        subject={"type": "adult_person", "gender": "man"},
        action={"type": "standing", "relation": "near"},
        reference={
            "object": "car", "object_labels": {ADE["car"]},
            "scale_range": (0.7, 1.7),
        },
        constraints=make_constraints(
            RELATIVE_STANDING_WEIGHTS, "standing", "proximity"
        ),
        metadata={
            "context_type": "object_relative", "ground_labels": OUTDOOR_GROUND,
            **RELATIVE_SPATIAL_RULES,
        },
    ),
    ContextSpecification(
        spec_id="spec_005",
        context_type="physical_interaction",
        prompt=(
            "A photorealistic adult woman sitting upright on the visible chair, bench, or "
            "seat. Hips on the seat, back toward any backrest, knees and feet toward the open "
            "side. Complete head, torso, two arms, two legs, and both feet visible. Natural "
            "anatomy, scale, perspective, lighting, and shadows."
        ),
        negative_prompt=NEGATIVE_PROMPT,
        subject={"type": "adult_person", "gender": "woman"},
        action={"type": "sitting", "relation": "sitting_on"},
        reference={
            "object": "bench_or_seat",
            "object_labels": {ADE["bench"], ADE["chair"], ADE["seat"]},
            "scale_range": (0.7, 2.2),
        },
        constraints=make_constraints(
            SITTING_INTERACTION_WEIGHTS, "sitting", "overlap"
        ),
        metadata={"context_type": "physical_interaction", **INTERACTION_SPATIAL_RULES},
    ),
    ContextSpecification(
        spec_id="spec_006",
        context_type="physical_interaction",
        prompt=(
            "A photorealistic adult man sitting upright on the visible sofa or armchair. "
            "Hips on the cushion, back toward the backrest, knees and feet toward the open "
            "side. Complete head, torso, two arms, two legs, and both feet visible. Natural "
            "anatomy, scale, perspective, lighting, and shadows."
        ),
        negative_prompt=NEGATIVE_PROMPT,
        subject={"type": "adult_person", "gender": "man"},
        action={"type": "sitting", "relation": "sitting_on"},
        reference={
            "object": "sofa_or_armchair",
            "object_labels": {ADE["sofa"], ADE["armchair"]},
            "scale_range": (0.5, 1.6),
        },
        constraints=make_constraints(
            SITTING_INTERACTION_WEIGHTS, "sitting", "overlap"
        ),
        metadata={"context_type": "physical_interaction", **INTERACTION_SPATIAL_RULES},
    ),
]


### 5.2. Validation và lookup

Kiểm tra uniqueness và semantic consistency trước khi proposal search bắt đầu.


In [ ]:
def validate_context_specifications(specifications):
    spec_ids = [spec.spec_id for spec in specifications]
    duplicates = sorted({spec_id for spec_id in spec_ids if spec_ids.count(spec_id) > 1})
    if duplicates:
        raise ValueError(f"spec_id phải duy nhất; bị trùng: {duplicates}")
    for specification in specifications:
        constraint_names = [c.name for c in specification.constraints]
        duplicate_constraints = sorted({
            name for name in constraint_names if constraint_names.count(name) > 1
        })
        if duplicate_constraints:
            raise ValueError(
                f"{specification.spec_id}: constraint bị trùng {duplicate_constraints}"
            )
        strategy = specification.metadata.get("proposal_strategy")
        if strategy not in {"free_space", "object_relative", "object_interaction"}:
            raise ValueError(f"{specification.spec_id}: proposal_strategy không hợp lệ")
        pose_constraints = [
            constraint for constraint in specification.constraints
            if constraint.name == "pose_contact" and constraint.importance == "soft"
        ]
        if len(pose_constraints) != 1:
            raise ValueError(f"{specification.spec_id}: cần đúng một pose_contact soft constraint")
        pose_mode = pose_constraints[0].params.get("pose_mode")
        if pose_mode not in {"standing", "sitting"}:
            raise ValueError(f"{specification.spec_id}: pose_mode không hợp lệ")
        relation_modes = {
            constraint.params.get("mode")
            for constraint in specification.constraints
            if constraint.name == "relation"
        }
        if relation_modes - {"proximity", "overlap"}:
            raise ValueError(f"{specification.spec_id}: relation mode không hợp lệ")
        if len(relation_modes) > 1:
            raise ValueError(f"{specification.spec_id}: relation mode không nhất quán")
        if relation_modes and specification.reference is None:
            raise ValueError(f"{specification.spec_id}: relation cần reference object")


validate_context_specifications(CONTEXT_SPECIFICATIONS)
CONTEXT_SPEC_BY_ID = {spec.spec_id: spec for spec in CONTEXT_SPECIFICATIONS}


def get_context_spec(spec_id):
    """Trả về ContextSpecification theo spec_id."""
    try:
        return CONTEXT_SPEC_BY_ID[spec_id]
    except KeyError as exc:
        raise KeyError(f"Không tìm thấy context specification: {spec_id}") from exc


def get_constraint_spec(specification, name, importance=None):
    """Truy cập constraint theo semantic name thay vì context category."""
    matches = [
        constraint for constraint in specification.constraints
        if constraint.name == name
        and (importance is None or constraint.importance == importance)
    ]
    if not matches:
        return None
    if len(matches) > 1 and importance is None:
        raise ValueError(f"Constraint {name} cần chỉ rõ importance")
    return matches[0]


def hard_threshold(specification, name):
    constraint = get_constraint_spec(specification, name, "hard")
    if constraint is None:
        raise KeyError(f"{specification.spec_id} thiếu hard constraint {name}")
    return constraint.threshold


pd.DataFrame([{
    "spec_id": spec.spec_id,
    "context_type": spec.context_type,
    "subject": spec.subject["gender"],
    "action": (spec.action or {}).get("type"),
    "reference_object": (spec.reference or {}).get("object", ""),
    "num_constraints": len(spec.constraints),
} for spec in CONTEXT_SPECIFICATIONS])


## 6. Đọc và chuẩn hóa dữ liệu ADE20K

Dataset streaming có thể trả ảnh dưới dạng PIL, path hoặc dictionary. Các hàm dưới đây dò field ảnh/mask theo mode, chuyển ảnh về RGB và kiểm tra semantic ID nằm trong khoảng `0..150`.

In [ ]:
def pil_from_value(value):
    if isinstance(value, Image.Image):
        return value
    if isinstance(value, dict) and "path" in value:
        return Image.open(value["path"])
    if isinstance(value, str):
        return Image.open(value)
    raise TypeError(f"Không thể chuyển value thành PIL image: {type(value)}")

def detect_image_and_mask(example):
    """Dò field ảnh RGB và semantic mask từ một example của dataset."""
    image_candidates = []
    mask_candidates = []

    for key, value in example.items():
        try:
            pil = pil_from_value(value)
        except Exception:
            continue

        if pil.mode in {"L", "P", "I", "I;16"}:
            mask_candidates.append((key, pil))
        else:
            image_candidates.append((key, pil))

    if not image_candidates or not mask_candidates:
        raise ValueError(
            f"Không dò được image/mask. Fields: {list(example.keys())}"
        )

    # Ảnh RGB lớn nhất; mask grayscale lớn nhất.
    image_key, image = max(
        image_candidates,
        key=lambda item: item[1].width * item[1].height,
    )
    mask_key, mask = max(
        mask_candidates,
        key=lambda item: item[1].width * item[1].height,
    )

    return image_key, image.convert("RGB"), mask_key, mask

def normalize_semantic_mask(mask_image):
    """Chuyển mask về mảng int32 hai chiều và kiểm tra miền class ID."""
    arr = np.array(mask_image)

    if arr.ndim == 3:
        arr = arr[..., 0]

    arr = arr.astype(np.int32)

    if arr.min() < 0 or arr.max() > 150:
        raise ValueError(
            "Semantic mask có giá trị ngoài khoảng SceneParse150 "
            f"0..150: min={arr.min()}, max={arr.max()}"
        )

    return arr


## 7. Tạo stream và chuẩn hóa từng example

`prepare_stream()` tạo stream có shuffle xác định bởi seed. `process_example()` trả về `(image_key, image, mask_key, seg)` và resize semantic mask bằng nearest-neighbor nếu kích thước chưa khớp ảnh nguồn.

In [ ]:
def prepare_stream(seed_offset=0):
    """Tạo ADE20K stream có thứ tự shuffle tái lập được."""
    dataset = load_dataset(
        "parquet",
        data_files={"validation": ADE20K_VALIDATION_PARQUET},
        split="validation",
        streaming=True,
    )
    return dataset.shuffle(
        seed=CONFIG["dataset_seed"] + seed_offset,
        buffer_size=CONFIG["shuffle_buffer_size"],
    )

def process_example(example):
    """Trả về ảnh RGB và semantic mask cùng kích thước."""
    image_key, image, mask_key, mask_image = detect_image_and_mask(example)
    seg = normalize_semantic_mask(mask_image)

    if seg.shape != (image.height, image.width):
        seg = np.array(
            Image.fromarray(seg.astype(np.uint8)).resize(
                image.size, Image.Resampling.NEAREST
            )
        ).astype(np.int32)

    return image_key, image, mask_key, seg

## 8. Phân tích semantic layout

Candidate được đánh giá trực tiếp trên semantic mask, không dựa vào caption. Trước hết notebook xây dựng các phép toán dùng chung: connected component, bounding box, integral image và các chỉ số ground/obstacle/sky cho vùng người dự kiến.

In [ ]:
def class_mask(seg, labels):
    labels = list(labels)
    return np.isin(seg, labels)

def area_ratio(mask):
    return float(mask.mean())

def largest_component(binary):
    labeled, count = ndimage.label(binary)
    if count == 0:
        return None, 0

    sizes = ndimage.sum(
        binary,
        labeled,
        index=np.arange(1, count + 1),
    )
    best_label = int(np.argmax(sizes)) + 1
    component = labeled == best_label
    return component, int(sizes[best_label - 1])

def bbox_from_mask(binary):
    ys, xs = np.where(binary)
    if len(xs) == 0:
        return None
    return (
        int(xs.min()),
        int(ys.min()),
        int(xs.max()) + 1,
        int(ys.max()) + 1,
    )

def touches_border(box, W, H, margin=0.025):
    x1, y1, x2, y2 = box
    return (
        x1 <= margin * W
        or y1 <= margin * H
        or x2 >= (1 - margin) * W
        or y2 >= (1 - margin) * H
    )

def integral_image(binary):
    return binary.astype(np.int32).cumsum(0).cumsum(1)

def rect_sum(integral, x1, y1, x2, y2):
    H, W = integral.shape
    x1, x2 = max(0, x1), min(W, x2)
    y1, y2 = max(0, y1), min(H, y2)
    if x1 >= x2 or y1 >= y2:
        return 0

    total = integral[y2 - 1, x2 - 1]
    if x1 > 0:
        total -= integral[y2 - 1, x1 - 1]
    if y1 > 0:
        total -= integral[y1 - 1, x2 - 1]
    if x1 > 0 and y1 > 0:
        total += integral[y1 - 1, x1 - 1]
    return int(total)


### 8.1. SpatialProposal schema

`SpatialProposal` là đơn vị nối specification, background và vùng chỉnh sửa. Một background có thể trả về nhiều proposal; baseline hiện tạo đúng một proposal (`p00`) để giữ nguyên chi phí. Geometry functions phía dưới vẫn giữ thuật toán cũ và chỉ thay lớp đầu ra. Reference có thể được biểu diễn bằng box, mask path, point hoặc object identity; schema không ép mọi detector phải sinh box.

`proposal_feasibility_score` chỉ mô tả mức hợp lý hình học trước generation. Đây không phải dự đoán chất lượng ảnh sinh.


In [ ]:
@dataclass
class SpatialProposal:
    proposal_id: str
    spec_id: str
    background_id: str
    target_box: tuple[int, int, int, int]
    target_mask_path: str
    reference_box: tuple[int, int, int, int] | None = None
    reference_mask_path: str | None = None
    reference_point: tuple[int, int] | None = None
    reference_identity: str | None = None
    expected_person_box: tuple[int, int, int, int] | None = None
    expected_scale: float | None = None
    ground_line: float | None = None
    feasibility_components: dict[str, float | None] = field(default_factory=dict)
    proposal_feasibility_score: float | None = None
    metadata: dict = field(default_factory=dict)

    def __post_init__(self):
        image_size = (
            int(self.metadata.get("image_width", 0)),
            int(self.metadata.get("image_height", 0)),
        )
        validate_box(self.target_box, image_size, "target_box")
        if self.expected_person_box is not None:
            validate_box(self.expected_person_box, image_size, "expected_person_box")
        if self.reference_box is not None:
            validate_box(self.reference_box, image_size, "reference_box")
        if self.reference_point is not None:
            validate_point(self.reference_point, image_size, "reference_point")
        for name, value in [
            ("reference_mask_path", self.reference_mask_path),
            ("reference_identity", self.reference_identity),
        ]:
            if value is not None and (not isinstance(value, str) or not value.strip()):
                raise ValueError(f"{name} phải là chuỗi không rỗng hoặc None")

        if self.proposal_feasibility_score is not None:
            validate_unit_interval(
                self.proposal_feasibility_score, "proposal_feasibility_score"
            )
        for name, value in self.feasibility_components.items():
            if value is not None:
                validate_unit_interval(value, f"feasibility_components[{name}]")

        specification = get_context_spec(self.spec_id)
        reference_representations = [
            self.reference_box, self.reference_mask_path,
            self.reference_point, self.reference_identity,
        ]
        if (specification.reference is not None
                and not any(value is not None for value in reference_representations)):
            raise ValueError(
                f"{self.proposal_id} thiếu reference representation "
                "(box, mask, point hoặc identity)"
            )


def validate_unit_interval(value, name):
    value = float(value)
    if not 0.0 <= value <= 1.0:
        raise ValueError(f"{name} phải nằm trong [0, 1], nhận được {value}")


def validate_box(box, image_size, name):
    if not isinstance(box, tuple) or len(box) != 4:
        raise ValueError(f"{name} phải là tuple (x1, y1, x2, y2)")
    if not all(isinstance(value, int) for value in box):
        raise ValueError(f"{name} chỉ được chứa số nguyên")
    x1, y1, x2, y2 = box
    width, height = image_size
    if width <= 0 or height <= 0:
        raise ValueError("metadata phải chứa image_width và image_height hợp lệ")
    if x2 <= x1 or y2 <= y1:
        raise ValueError(f"{name} không có diện tích dương: {box}")
    if x1 < 0 or y1 < 0 or x2 > width or y2 > height:
        raise ValueError(f"{name} nằm ngoài ảnh {image_size}: {box}")


def validate_point(point, image_size, name):
    if not isinstance(point, tuple) or len(point) != 2:
        raise ValueError(f"{name} phải là tuple (x, y)")
    if not all(isinstance(value, int) for value in point):
        raise ValueError(f"{name} chỉ được chứa số nguyên")
    x, y = point
    width, height = image_size
    if not (0 <= x < width and 0 <= y < height):
        raise ValueError(f"{name} nằm ngoài ảnh {image_size}: {point}")


def normalize_id_component(value):
    normalized = "".join(
        character if character.isalnum() or character in {"-", "_"} else "_"
        for character in str(value).strip()
    ).strip("_")
    if not normalized:
        raise ValueError("ID component không được rỗng")
    return normalized


def make_proposal_id(spec_id, background_id, proposal_index=0):
    return (
        f"{normalize_id_component(spec_id)}_"
        f"{normalize_id_component(background_id)}_p{int(proposal_index):02d}"
    )


def normalize_feasibility_score(raw_score):
    """Ánh xạ đơn điệu score geometry cũ vào [0, 1], không đổi thứ tự heap."""
    nonnegative = max(0.0, float(raw_score))
    return nonnegative / (1.0 + nonnegative)


def validate_unique_proposal_ids(proposals):
    proposal_ids = [proposal.proposal_id for proposal in proposals]
    duplicates = sorted({value for value in proposal_ids if proposal_ids.count(value) > 1})
    if duplicates:
        raise ValueError(f"proposal_id bị trùng: {duplicates}")


def stable_json(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def spatial_proposal_to_dict(proposal):
    """Public serializer cho generation/evaluation consumers bên ngoài notebook."""
    metadata = proposal.metadata
    specification = get_context_spec(proposal.spec_id)
    action = specification.action or {}
    reference = specification.reference or {}
    return {
        "proposal_id": proposal.proposal_id,
        "spec_id": proposal.spec_id,
        "context_type": specification.context_type,
        "relation": action.get("relation", ""),
        "reference_object": reference.get("object", ""),
        "prompt": specification.prompt,
        "negative_prompt": specification.negative_prompt,
        "background_id": proposal.background_id,
        "target_box": proposal.target_box,
        "target_mask_path": proposal.target_mask_path,
        "reference_box": proposal.reference_box,
        "reference_mask_path": proposal.reference_mask_path,
        "reference_point": proposal.reference_point,
        "reference_identity": proposal.reference_identity,
        "expected_person_box": proposal.expected_person_box,
        "expected_scale": proposal.expected_scale,
        "ground_line": proposal.ground_line,
        "proposal_feasibility_score": proposal.proposal_feasibility_score,
        "feasibility_components": dict(proposal.feasibility_components),
        "metadata": dict(metadata),
        "heap_rank": metadata.get("heap_rank"),
    }


### 8.2. CandidateState schema

`CandidateState` định danh một lần generation bằng `spatial proposal × latent seed`; `candidate_id` không chứa allocation round. Adaptive budget dùng `search_round=0/1/2` như metadata. `parent_candidate_id` vẫn là `None` vì v1 chưa có latent perturbation hay quan hệ refinement cha–con.

In [ ]:
@dataclass
class CandidateState:
    candidate_id: str
    spec_id: str
    background_id: str
    proposal_id: str
    seed: int
    search_round: int = 0
    parent_candidate_id: str | None = None
    generation_params: dict = field(default_factory=dict)
    output_path: str | None = None

    def __post_init__(self):
        self.seed = int(self.seed)
        self.search_round = int(self.search_round)
        if self.seed < 0:
            raise ValueError("seed không được âm")
        if self.search_round < 0:
            raise ValueError("search_round không được âm")
        if (self.parent_candidate_id is not None
                and (not isinstance(self.parent_candidate_id, str)
                     or not self.parent_candidate_id.strip())):
            raise ValueError("parent_candidate_id phải là chuỗi không rỗng hoặc None")
        expected_id = build_candidate_id(self.proposal_id, self.seed)
        if self.candidate_id != expected_id:
            raise ValueError(
                f"candidate_id không khớp state: {self.candidate_id} != {expected_id}"
            )
        get_context_spec(self.spec_id)
        stable_json(self.generation_params)


def build_candidate_id(proposal_id, seed):
    proposal_component = normalize_id_component(proposal_id)
    return f"{proposal_component}_s{int(seed)}"


def validate_candidate_states(states):
    candidate_ids = [state.candidate_id for state in states]
    seen = set()
    duplicates = set()
    for candidate_id in candidate_ids:
        if candidate_id in seen:
            duplicates.add(candidate_id)
        seen.add(candidate_id)
    if duplicates:
        raise ValueError(f"candidate_id bị trùng: {sorted(duplicates)}")
    missing_parents = sorted({
        state.parent_candidate_id for state in states
        if state.parent_candidate_id is not None
        and state.parent_candidate_id not in seen
    })
    if missing_parents:
        raise ValueError(f"parent_candidate_id không tồn tại: {missing_parents}")


def candidate_state_to_dict(state, proposal, specification):
    if state.proposal_id != proposal.proposal_id:
        raise ValueError("CandidateState và SpatialProposal không cùng proposal_id")
    if state.spec_id != proposal.spec_id or state.spec_id != specification.spec_id:
        raise ValueError("CandidateState, SpatialProposal và ContextSpecification lệch spec_id")
    if state.background_id != proposal.background_id:
        raise ValueError("CandidateState và SpatialProposal lệch background_id")

    return {
        "candidate_id": state.candidate_id,
        "proposal_id": state.proposal_id,
        "spec_id": state.spec_id,
        "context_type": specification.context_type,
        "background_id": state.background_id,
        "seed": state.seed,
        "search_round": state.search_round,
        "parent_candidate_id": state.parent_candidate_id,
        "generation_params": dict(state.generation_params),
        "output_path": state.output_path,
        "proposal_feasibility_score": proposal.proposal_feasibility_score,
        "heap_rank": proposal.metadata.get("heap_rank"),
    }


### 8.3. Đánh giá vùng người

`make_person_box()` dựng box có kích thước tương đối theo ảnh và dịch box vào trong biên thay vì crop. `evaluate_person_box()` đo tiếp xúc chân–ground, tỷ lệ obstacle và tỷ lệ sky.

In [ ]:
def make_person_box(anchor_x, anchor_y, W, H, width_ratio=0.18, height_ratio=0.62):
    person_width = int(width_ratio * W)
    person_height = int(height_ratio * H)

    x1 = int(anchor_x - person_width / 2)
    x2 = x1 + person_width

    y2 = int(anchor_y)
    y1 = y2 - person_height

    # Dịch box vào trong ảnh thay vì crop trực tiếp nhằm giữ nguyên kích thước mask khi gần mép.
    if x1 < 0:
        x2 -= x1
        x1 = 0

    if x2 > W:
        shift = x2 - W
        x1 -= shift
        x2 = W

    if y1 < 0:
        y2 -= y1
        y1 = 0

    if y2 > H:
        shift = y2 - H
        y1 -= shift
        y2 = H

    return (
        max(0, int(x1)),
        max(0, int(y1)),
        min(W, int(x2)),
        min(H, int(y2)),
    )

def evaluate_person_box(seg, box, allowed_ground_labels, reference_mask=None):
    H, W = seg.shape
    x1, y1, x2, y2 = box
    area = max(1, (x2 - x1) * (y2 - y1))

    ground = class_mask(seg, allowed_ground_labels)
    ground_integral = integral_image(ground)

    # Dải chân dưới cùng của box phải tiếp xúc ground.
    foot_h = max(2, int(0.08 * (y2 - y1)))
    foot_ground = rect_sum(ground_integral, x1, y2 - foot_h, x2, y2) / max(1, (x2 - x1) * foot_h)

    allowed = ground.copy()
    allowed |= seg == 0

    if reference_mask is not None:
        allowed |= reference_mask

    obstacle = ~allowed
    obstacle_integral = integral_image(obstacle)

    obstacle_ratio = rect_sum(obstacle_integral, x1, y1, x2, y2) / area

    sky_ratio = float((seg[y1:y2, x1:x2] == ADE["sky"]).mean())

    return {
        "foot_ground_ratio": foot_ground,
        "obstacle_ratio": obstacle_ratio,
        "sky_ratio": sky_ratio,
    }


### 8.4. Free placement

Quét một lưới anchor thưa ở bên trái hoặc trung tâm ảnh. Candidate chỉ được giữ khi chân tiếp xúc ground, obstacle thấp và vùng phía trên không chứa quá nhiều sky. Với `center_foreground`, notebook ưu tiên cột khả dụng gần trục 50% nhất rồi mới so geometry score, tránh mask lệch trái chỉ vì ground score cao hơn một chút.

In [ ]:
def best_free_box(seg, relation, ground_labels):
    """Chọn box đứng tự do tốt nhất trên lưới anchor thưa."""
    H, W = seg.shape
    ground = class_mask(seg, ground_labels)

    if relation == "left_side":
        x_fracs = np.linspace(0.16, 0.38, 5)
        desired_x = 0.27
    elif pose_mode == "standing":
        x_fracs = np.linspace(0.40, 0.60, 5)
        desired_x = 0.50

    y_fracs = np.linspace(0.78, 0.96, 5)
    candidates = []

    for xf in x_fracs:
        for yf in y_fracs:
            x = int(xf * W)
            y = int(yf * H)

            if not ground[y, x]:
                continue

            box = make_person_box(x, y, W, H,
                                  width_ratio=CONFIG["standing_mask_width_ratio"],
                                  height_ratio=CONFIG["standing_mask_height_ratio"],
                                  )
            metrics = evaluate_person_box(seg, box, ground_labels)
            box_center_x = (box[0] + box[2]) / (2 * W)
            alignment = max(0.0, 1.0 - abs(box_center_x - desired_x) / 0.15)
            metrics["horizontal_alignment_score"] = alignment

            if metrics["foot_ground_ratio"] < 0.35:
                continue
            if metrics["obstacle_ratio"] > 0.32:
                continue
            if metrics["sky_ratio"] > 0.55:
                continue

            score = (
                1.6 * metrics["foot_ground_ratio"]
                + 1.2 * (1 - metrics["obstacle_ratio"])
                + 0.4 * (1 - metrics["sky_ratio"])
                + 0.3 * alignment
            )

            candidates.append((score, box, metrics, alignment))

    if not candidates:
        return None

    if relation == "center_foreground":
        # Ưu tiên cột gần trục giữa nhất; geometry score chỉ phá hòa trong cột đó.
        best_alignment = max(item[3] for item in candidates)
        candidates = [item for item in candidates if item[3] >= best_alignment - 1e-6]

    score, box, metrics, _ = max(candidates, key=lambda item: item[0])
    return score, box, metrics


### 8.5. Object-relative placement

Lấy connected component lớn nhất của reference object và thử anchor ở hai bên. Chế độ relaxed chấp nhận table sát biên, nhưng chỉ giữ phía còn đủ chỗ cho tâm người và giới hạn phần mask chồng lên table; nếu cả hai phía đều không hợp lệ thì ảnh vẫn bị loại.

In [ ]:
def best_relative_box(seg, object_labels, ground_labels, relaxed=False):
    """Chọn box đứng cạnh connected component lớn nhất của object."""
    H, W = seg.shape
    object_binary = class_mask(seg, object_labels)
    component, pixels = largest_component(object_binary)

    if component is None:
        return None

    component_ratio = pixels / (W * H)
    min_component_ratio = 0.004 if relaxed else 0.008
    if not (min_component_ratio <= component_ratio <= 0.35):
        return None

    object_box = bbox_from_mask(component)
    if object_box is None or (not relaxed and touches_border(object_box, W, H)):
        return None

    x1, y1, x2, y2 = object_box

    if relaxed:
        min_foot = CONFIG["relaxed_min_foot_ground_ratio"]
        max_obstacle = CONFIG["relaxed_max_obstacle_ratio"]
        max_distance = CONFIG["relaxed_max_relative_distance"]
        anchor_distances = [0.06, 0.10, 0.15, 0.21, 0.28]
        search_up_ratio = 0.24
        search_down_ratio = 0.32
    else:
        min_foot = CONFIG["strict_min_foot_ground_ratio"]
        max_obstacle = CONFIG["strict_max_obstacle_ratio"]
        max_distance = CONFIG["strict_max_relative_distance"]
        anchor_distances = [0.08, 0.14, 0.20, 0.27]
        search_up_ratio = 0.20
        search_down_ratio = 0.28

    anchors = []
    for distance_ratio in anchor_distances:
        anchors.extend([
            (max(0, x1 - int(distance_ratio * W)),
             min(H - 1, y2 + int(0.05 * H)), "left"),
            (min(W - 1, x2 + int(distance_ratio * W)),
             min(H - 1, y2 + int(0.05 * H)), "right"),
        ])

    ground = class_mask(seg, ground_labels)
    candidates = []

    for anchor_x, approximate_y, side in anchors:
        search_y1 = max(0, approximate_y - int(search_up_ratio * H))
        search_y2 = min(H, approximate_y + int(search_down_ratio * H))

        ground_rows = np.where(ground[search_y1:search_y2, anchor_x])[0]
        if len(ground_rows) == 0:
            continue

        anchor_y = search_y1 + int(ground_rows[-1])
        box = make_person_box(anchor_x, anchor_y, W, H,
                              width_ratio=CONFIG["relative_mask_width_ratio"],
                              height_ratio=CONFIG["relative_mask_height_ratio"],
                              )

        # Anchor bị clamp ở biên có thể đẩy box sang phía sai của object.
        person_center_x = (box[0] + box[2]) / 2
        wrong_side = (side == "left" and person_center_x >= x1) or (
            side == "right" and person_center_x <= x2
        )
        if wrong_side:
            continue

        metrics = evaluate_person_box(
            seg, box, ground_labels, reference_mask=component
        )
        person_area = max(1, (box[2] - box[0]) * (box[3] - box[1]))
        reference_overlap = rect_sum(
            integral_image(component), box[0], box[1], box[2], box[3]
        ) / person_area
        metrics["reference_overlap_ratio"] = reference_overlap

        if metrics["foot_ground_ratio"] < min_foot:
            continue
        if metrics["obstacle_ratio"] > max_obstacle:
            continue
        if reference_overlap > (0.15 if relaxed else 0.08):
            continue

        object_center_x = (x1 + x2) / 2
        normalized_distance = abs(person_center_x - object_center_x) / W

        if normalized_distance > max_distance:
            continue

        score = (
            1.4 * metrics["foot_ground_ratio"]
            + 1.0 * (1 - metrics["obstacle_ratio"])
            + 0.8 * component_ratio
            + 0.3 * (1 - normalized_distance)
            - 0.5 * reference_overlap
        )

        if not relaxed:
            score += 0.15

        candidates.append(
            (score, box, object_box, side, metrics, component_ratio)
        )

    if not candidates:
        return None

    return max(candidates, key=lambda item: item[0])


### 8.6. Physical interaction

Với hành động ngồi, target box được neo gần mặt ghế và chồng một phần lên reference object để mô hình có đủ không gian tạo thân, hông và chân. Strict search ưu tiên ghế đủ lớn, không sát biên; relaxed search chỉ nới các ngưỡng hình học khi chưa tìm đủ candidate.

In [ ]:
def best_interaction_box(seg, object_labels, relaxed=False):
    """Đặt box ngồi chồng có kiểm soát lên reference object."""
    H, W = seg.shape
    object_binary = class_mask(seg, object_labels)
    component, pixels = largest_component(object_binary)

    if component is None:
        return None

    component_ratio = pixels / (W * H)
    min_component_ratio = 0.005 if relaxed else 0.010
    if not (min_component_ratio <= component_ratio <= 0.35):
        return None

    object_box = bbox_from_mask(component)
    if not relaxed and touches_border(object_box, W, H):
        return None

    x1, y1, x2, y2 = object_box
    object_w = x2 - x1
    object_h = y2 - y1

    min_width = (0.06 if relaxed else 0.10) * W
    min_height = (0.04 if relaxed else 0.06) * H
    if object_w < min_width or object_h < min_height:
        return None

    anchor_x = int((x1 + x2) / 2)
    # Neo gần mặt ghế để box bao phủ thân trên và phần chân phía trước ghế.
    seat_y = int(y1 + 0.70 * object_h)
    anchor_y = min(H - 1, max(seat_y + int(0.06 * H), y2 + int(0.10 * H)))

    adaptive_width_ratio = max(
        CONFIG["sitting_mask_width_ratio"],
        min(0.32, object_w / W * 1.05),
    )

    box = make_person_box(
        anchor_x,
        anchor_y,
        W,
        H,
        width_ratio=adaptive_width_ratio,
        height_ratio=CONFIG["sitting_mask_height_ratio"],
    )

    metrics = evaluate_person_box(seg, box, ALL_GROUND, reference_mask=component)

    overlap = rect_sum(
        integral_image(component), box[0], box[1], box[2], box[3]
    ) / max(1, (box[2] - box[0]) * (box[3] - box[1]))

    metrics["reference_overlap_ratio"] = overlap

    if overlap < (0.004 if relaxed else 0.008):
        return None
    if metrics["obstacle_ratio"] > (0.62 if relaxed else 0.52):
        return None

    score = (1.2 * component_ratio + 1.0 * overlap + 0.7 * (1 - metrics["obstacle_ratio"]))

    return (
        score,
        box,
        object_box,
        "overlap",
        metrics,
        component_ratio,
    )


### 8.7. General person-canvas mask

Mọi context dùng cùng một person-canvas mask bo góc; mask không mã hóa trước pose đứng/ngồi. Spatial rule `allow_reference_occlusion` quyết định có bảo vệ reference object hay cho phép cơ thể che khuất nó.


In [ ]:
def add_shadow_space(box, H, extension_ratio=None):
    """Chừa một dải nhỏ dưới chân để mô hình có thể tạo contact shadow."""
    if extension_ratio is None:
        extension_ratio = CONFIG["shadow_extension_ratio"]
    x1, y1, x2, y2 = box
    extension = int(extension_ratio * max(1, y2 - y1))
    return x1, y1, x2, min(H, y2 + extension)


def build_person_canvas_mask(image_size, box):
    """Tạo person-canvas bo góc liên tục, dùng chung cho mọi pose."""
    W, H = image_size
    x1, y1, x2, y2 = box
    w, h = x2 - x1, y2 - y1
    mask = Image.new("L", (W, H), 0)
    radius = min(w // 2, int(CONFIG["mask_corner_radius_ratio"] * min(w, h)))
    ImageDraw.Draw(mask).rounded_rectangle(
        (x1, y1, x2 - 1, y2 - 1), radius=radius, fill=255
    )
    return np.asarray(mask) > 0


def build_target_mask(seg, target_box, spec):
    """Tạo hard mask liên tục và giữ nguyên reference object khi không có occlusion."""
    H, W = seg.shape
    target = build_person_canvas_mask((W, H), target_box)

    if spec.metadata.get("allow_reference_occlusion", False):
        # Interaction cần cho phép cơ thể che khuất một phần ghế trong canvas.
        return target.astype(np.uint8) * 255

    x1, _, x2, y2 = target_box
    _, _, _, shadow_y2 = add_shadow_space(target_box, H)
    center_x = (x1 + x2) // 2
    shadow_half_w = max(1, int(0.30 * (x2 - x1)))
    target[y2:shadow_y2, center_x-shadow_half_w:center_x+shadow_half_w] = True

    object_labels = (spec.reference or {}).get("object_labels")
    if object_labels:
        reference = class_mask(seg, object_labels)
        reference, _ = largest_component(reference)
        if reference is not None:
            protected = ndimage.binary_dilation(
                reference, iterations=CONFIG["reference_protection_dilation"]
            )
            target &= ~protected

    return target.astype(np.uint8) * 255


## 9. Tạo SpatialProposal trên stream

Mỗi background được đối chiếu với sáu specification và trả về `list[SpatialProposal]`; baseline hiện có tối đa một phần tử. Min-heap chỉ giữ top-`K` theo feasibility, tránh lưu toàn bộ stream trong RAM. Workflow chạy strict trước rồi relaxed supplement cho specification còn thiếu proposal.


In [ ]:
def push_top_k(heap, item, k):
    if len(heap) < k:
        heapq.heappush(heap, item)
    else:
        heapq.heappushpop(heap, item)


def feasibility_components(specification, metrics, semantic_area_ratio):
    """Chuẩn hóa các geometry measurements thành components cùng chiều [0, 1]."""
    reference_overlap = metrics.get("reference_overlap_ratio")
    rules = specification.metadata
    reference_relation = rules.get("reference_relation")
    return {
        "ground_support": (
            float(metrics["foot_ground_ratio"])
            if rules.get("ground_contact", False) else None
        ),
        "obstacle_clearance": 1.0 - float(metrics["obstacle_ratio"]),
        "sky_clearance": (
            1.0 - float(metrics["sky_ratio"])
            if rules.get("sky_clearance", False) else None
        ),
        "horizontal_alignment": (
            float(metrics["horizontal_alignment_score"])
            if "horizontal_alignment_score" in metrics else None
        ),
        "reference_clearance": (
            1.0 - float(reference_overlap)
            if reference_relation == "clearance" and reference_overlap is not None else None
        ),
        "reference_overlap": (
            float(reference_overlap)
            if reference_relation == "overlap" and reference_overlap is not None else None
        ),
        "semantic_support": float(semantic_area_ratio),
    }


def create_spatial_proposal(
    spec, background_id, image_size, raw_score, target_box,
    reference_box, placement_side, metrics, semantic_area_ratio,
    selection_mode, proposal_index=0,
):
    """Đóng gói kết quả geometry hiện tại; baseline tạo một proposal p00."""
    width, height = image_size
    proposal_id = make_proposal_id(spec.spec_id, background_id, proposal_index)
    target_box = tuple(int(value) for value in target_box)
    reference_box = (
        tuple(int(value) for value in reference_box)
        if reference_box is not None else None
    )
    return SpatialProposal(
        proposal_id=proposal_id,
        spec_id=spec.spec_id,
        background_id=background_id,
        target_box=target_box,
        target_mask_path=f"masks/{proposal_id}.png",
        reference_box=reference_box,
        reference_mask_path=None,
        reference_point=None,
        reference_identity=(spec.reference or {}).get("object"),
        expected_person_box=target_box,
        expected_scale=(target_box[3] - target_box[1]) / height,
        ground_line=(
            float(target_box[3])
            if spec.metadata.get("ground_contact", False) else None
        ),
        feasibility_components=feasibility_components(
            spec, metrics, semantic_area_ratio
        ),
        proposal_feasibility_score=normalize_feasibility_score(raw_score),
        metadata={
            "image_width": width,
            "image_height": height,
            "placement_side": placement_side,
            "selection_mode": selection_mode,
            "semantic_area_ratio": float(semantic_area_ratio),
            "raw_geometry_score": float(raw_score),
            "geometry_metrics": {
                name: float(value) for name, value in metrics.items()
            },
        },
    )


def spec_match(spec, seg, background_id, relaxed=False):
    """Trả danh sách SpatialProposal phù hợp với một spec/background."""
    if (seg == PERSON_LABEL).any():
        return []

    height, width = seg.shape
    aspect = width / height
    selection_mode = "relaxed" if relaxed else "strict"
    action = spec.action or {}
    reference = spec.reference or {}
    ground_labels = spec.metadata.get("ground_labels", set())
    proposal_strategy = spec.metadata["proposal_strategy"]

    if relaxed:
        if width < 400 or height < 300 or not (0.90 <= aspect <= 2.50):
            return []
    elif width < 480 or height < 360 or not (1.05 <= aspect <= 2.20):
        return []

    if proposal_strategy == "free_space":
        ground_ratio = area_ratio(class_mask(seg, ground_labels))
        if ground_ratio < (0.07 if relaxed else 0.10):
            return []
        result = best_free_box(seg, action["relation"], ground_labels)
        if result is None:
            return []
        score, target_box, metrics = result
        proposal = create_spatial_proposal(
            spec, background_id, (width, height), score + 0.5 * ground_ratio,
            target_box, None, action["relation"], metrics, ground_ratio,
            selection_mode,
        )
        return [proposal]

    if proposal_strategy == "object_relative":
        result = best_relative_box(
            seg, reference["object_labels"], ground_labels, relaxed=relaxed
        )
        if result is None:
            return []
        score, target_box, reference_box, side, metrics, ratio = result
    elif proposal_strategy == "object_interaction":
        result = best_interaction_box(seg, reference["object_labels"], relaxed=relaxed)
        if result is None:
            return []
        score, target_box, reference_box, side, metrics, ratio = result
    else:
        raise ValueError(f"proposal_strategy không được hỗ trợ: {proposal_strategy}")

    return [create_spatial_proposal(
        spec, background_id, (width, height), score, target_box,
        reference_box, side, metrics, ratio, selection_mode,
    )]


### 9.1. Strict search với relaxed fallback

`run_candidate_search()` quản lý toàn bộ hai giai đoạn và trả về heap đã hoàn chỉnh. Strict và relaxed vẫn được ghi trong `selection_mode` để audit, nhưng không còn hai block thực thi rời nhau. Fingerprint ngăn cùng một ảnh bị thêm lại khi lượt relaxed dùng shuffle seed khác.


In [ ]:
def collect_candidates(heaps, specs, relaxed, seed_offset, counter, target_count):
    stream = prepare_stream(seed_offset=seed_offset)
    processed = 0
    start = time.time()
    mode = "relaxed" if relaxed else "strict"

    for example_index, example in enumerate(stream):
        if processed >= CONFIG["max_stream_examples"]:
            break

        _, image, _, seg = process_example(example)
        fingerprint = hashlib.blake2b(image.tobytes(), digest_size=8).hexdigest()
        background_id = f"bg_{fingerprint}"

        for spec in specs:
            spec_id = spec.spec_id
            if len(heaps[spec_id]) >= target_count:
                continue
            known_backgrounds = {
                item[2]["proposal"].background_id for item in heaps[spec_id]
            }
            if background_id in known_backgrounds:
                continue

            proposals = spec_match(
                spec, seg, background_id=background_id, relaxed=relaxed
            )
            for proposal in proposals:
                payload = {
                    "example_index": example_index,
                    "fingerprint": fingerprint,
                    "image": image.copy(),
                    "semantic_mask": seg.copy(),
                    "proposal": proposal,
                }
                push_top_k(
                    heaps[spec_id],
                    (proposal.proposal_feasibility_score, counter, payload),
                    CONFIG["top_k_candidates_per_spec"],
                )
                counter += 1

        processed += 1
        if processed % 100 == 0:
            counts = {spec.spec_id: len(heaps[spec.spec_id]) for spec in specs}
            print(f"{mode} processed={processed} proposals={counts} elapsed={time.time()-start:.1f}s")

        if all(len(heaps[spec.spec_id]) >= target_count for spec in specs):
            break

    return processed, counter


def run_candidate_search(specs):
    target_count = CONFIG["min_candidates_before_early_stop"]
    heaps = {spec.spec_id: [] for spec in specs}

    strict_processed, counter = collect_candidates(
        heaps, specs, relaxed=False, seed_offset=0,
        counter=0, target_count=target_count,
    )
    print("Strict processed examples:", strict_processed)

    underfilled = [
        spec for spec in specs
        if len(heaps[spec.spec_id]) < target_count
    ]
    if underfilled:
        print("Running relaxed supplement for:", [spec.spec_id for spec in underfilled])
        relaxed_processed, _ = collect_candidates(
            heaps, underfilled, relaxed=True, seed_offset=1,
            counter=counter, target_count=target_count,
        )
        print("Relaxed processed examples:", relaxed_processed)

    all_proposals = [
        item[2]["proposal"] for heap in heaps.values() for item in heap
    ]
    validate_unique_proposal_ids(all_proposals)

    final_counts = {spec_id: len(heap) for spec_id, heap in heaps.items()}
    print("Final proposal counts:", final_counts)
    shortfall = {
        spec_id: count for spec_id, count in final_counts.items()
        if count < target_count
    }
    if shortfall:
        print("WARNING - proposal shortfall:", shortfall)

    minimum = CONFIG["backgrounds_per_spec"]
    insufficient = {
        spec_id: count for spec_id, count in final_counts.items()
        if count < minimum
    }
    if insufficient:
        raise RuntimeError(
            f"Không đủ {minimum} proposal cho mỗi specification: {insufficient}"
        )
    return heaps


PROPOSAL_MANIFEST_COLUMNS = [
    "proposal_id", "spec_id", "context_type", "background_id", "heap_rank",
    "target_box", "expected_person_box", "reference_box",
    "target_mask_path", "reference_mask_path",
    "reference_point", "reference_identity",
    "proposal_feasibility_score", "feasibility_components",
]


if REBUILD_DATASET:
    top_heaps = run_candidate_search(CONTEXT_SPECIFICATIONS)


## 10. Chọn background và spatial proposal từ heap

Mỗi specification chọn bốn rank trải đều trên heap để tạo biến thiên proposal feasibility. Rank vẫn bắt đầu từ 1. Với baseline một proposal/background, thao tác này tương đương chọn bốn background; schema không giả định điều đó cho các phiên bản sau.


In [ ]:
def stratified_heap_ranks(heap_size, count):
    if heap_size < count:
        raise RuntimeError(f"Heap chỉ có {heap_size} candidate, cần ít nhất {count}.")
    ranks = np.rint(np.linspace(1, heap_size, count)).astype(int).tolist()
    return sorted(set(ranks))


def preview_candidates(heap, spec_id, cols=4, thumb=256):
    ranked = sorted(heap, key=lambda item: item[0], reverse=True)
    selected_ranks = stratified_heap_ranks(
        len(ranked), CONFIG["backgrounds_per_spec"]
    )
    label_h = 44
    rows = math.ceil(len(ranked) / cols)
    sheet = Image.new("RGB", (cols * thumb, rows * (thumb + label_h)), "white")
    draw_sheet = ImageDraw.Draw(sheet)

    for index, (score, _, payload) in enumerate(ranked):
        rank = index + 1
        proposal = payload["proposal"]
        image = payload["image"].copy()
        draw = ImageDraw.Draw(image)
        draw.rectangle(proposal.target_box, outline="red", width=max(2, image.width // 250))
        if proposal.reference_box is not None:
            draw.rectangle(proposal.reference_box, outline="lime", width=max(2, image.width // 250))
        image.thumbnail((thumb, thumb))

        x = (index % cols) * thumb
        y = (index // cols) * (thumb + label_h)
        canvas = Image.new("RGB", (thumb, thumb), "white")
        canvas.paste(image, ((thumb - image.width) // 2, (thumb - image.height) // 2))
        sheet.paste(canvas, (x, y))
        marker = "SELECTED" if rank in selected_ranks else ""
        draw_sheet.text((x + 6, y + thumb + 4), f"rank={rank} feasibility={score:.3f} {marker}", fill="black")
        draw_sheet.text(
            (x + 6, y + thumb + 22),
            f"{proposal.background_id} {proposal.metadata['selection_mode']}",
            fill="black",
        )

    preview_path = OUTPUT_DIR / "candidate_previews" / f"{spec_id}.png"
    sheet.save(preview_path)
    print(spec_id, "| selected ranks:", selected_ranks, "|", preview_path)
    display(sheet)


if REBUILD_DATASET and CREATE_PREVIEWS:
    for spec in CONTEXT_SPECIFICATIONS:
        preview_candidates(top_heaps[spec.spec_id], spec.spec_id)


### 10.1. Lưu proposal và dataset cache

`proposal_manifest.csv` lưu schema proposal chuẩn; `dataset_manifest.csv` bổ sung source paths và semantic metadata cần cho generation/evaluator. `proposal_search_audit.csv` lưu toàn bộ heap trước generation; `candidate_audit.csv` được tạo sau evaluator.


In [ ]:
if REBUILD_DATASET:
    selected_rows = []
    audit_rows = []
    selected_proposals = []

    for spec in CONTEXT_SPECIFICATIONS:
        spec_id = spec.spec_id
        action = spec.action or {}
        reference = spec.reference or {}
        ranked = sorted(top_heaps[spec_id], key=lambda item: item[0], reverse=True)
        selected_ranks = stratified_heap_ranks(
            len(ranked), CONFIG["backgrounds_per_spec"]
        )

        for rank, (_, _, payload) in enumerate(ranked, start=1):
            proposal = payload["proposal"]
            audit_rows.append({
                "proposal_id": proposal.proposal_id,
                "spec_id": spec_id,
                "background_id": proposal.background_id,
                "heap_rank": rank,
                "selected": int(rank in selected_ranks),
                "proposal_feasibility_score": proposal.proposal_feasibility_score,
                "raw_geometry_score": proposal.metadata["raw_geometry_score"],
                "example_index": payload["example_index"],
                "context_type": spec.context_type,
                "relation": action.get("relation", ""),
                "reference_object": reference.get("object", ""),
                "target_box": stable_json(proposal.target_box),
                "reference_box": stable_json(proposal.reference_box),
                "reference_point": stable_json(proposal.reference_point),
                "reference_identity": proposal.reference_identity,
                "feasibility_components": stable_json(proposal.feasibility_components),
                "placement_side": proposal.metadata["placement_side"],
                "semantic_area_ratio": proposal.metadata["semantic_area_ratio"],
                "selection_mode": proposal.metadata["selection_mode"],
                **proposal.metadata["geometry_metrics"],
            })

        for heap_rank in selected_ranks:
            _, _, selected = ranked[heap_rank - 1]
            proposal = selected["proposal"]
            image_path = BACKGROUND_DIR / f"{proposal.proposal_id}.jpg"
            semantic_path = SEMANTIC_DIR / f"{proposal.proposal_id}.png"
            mask_path = INPUT_DIR / proposal.target_mask_path

            proposal.metadata["heap_rank"] = heap_rank
            selected_proposals.append(proposal)

            selected["image"].save(image_path, quality=95)
            seg = selected["semantic_mask"].astype(np.uint8)
            Image.fromarray(seg).save(semantic_path)
            target_mask = Image.fromarray(
                build_target_mask(seg, proposal.target_box, spec)
            )
            target_mask.save(mask_path)

            selected_rows.append({
                "proposal_id": proposal.proposal_id,
                "spec_id": spec_id,
                "context_type": spec.context_type,
                "background_id": proposal.background_id,
                "heap_rank": heap_rank,
                "proposal_feasibility_score": proposal.proposal_feasibility_score,
                "target_box": stable_json(proposal.target_box),
                "expected_person_box": stable_json(proposal.expected_person_box),
                "reference_box": stable_json(proposal.reference_box),
                "target_mask_path": proposal.target_mask_path,
                "reference_mask_path": proposal.reference_mask_path,
                "reference_point": stable_json(proposal.reference_point),
                "reference_identity": proposal.reference_identity,
                "expected_scale": proposal.expected_scale,
                "ground_line": proposal.ground_line,
                "feasibility_components": stable_json(proposal.feasibility_components),
                "source_dataset": CONFIG["dataset_id"],
                "source_split": CONFIG["dataset_split"],
                "source_example_index": selected["example_index"],
                "relation": action.get("relation", ""),
                "reference_object": reference.get("object", ""),
                "object_label_ids": stable_json(sorted(reference.get("object_labels", []))),
                "reference_scale_range": stable_json(reference.get("scale_range")),
                "background_path": str(image_path.relative_to(INPUT_DIR)),
                "semantic_mask_path": str(semantic_path.relative_to(INPUT_DIR)),
                "prompt": spec.prompt,
            })

    validate_unique_proposal_ids(selected_proposals)
    dataset_df = pd.DataFrame(selected_rows)
    audit_df = pd.DataFrame(audit_rows)
    proposal_df = dataset_df[PROPOSAL_MANIFEST_COLUMNS].copy()

    dataset_df.to_csv(DATASET_MANIFEST_PATH, index=False, encoding="utf-8-sig")
    proposal_df.to_csv(PROPOSAL_MANIFEST_PATH, index=False, encoding="utf-8-sig")
    audit_df.to_csv(PROPOSAL_SEARCH_AUDIT_PATH, index=False, encoding="utf-8-sig")
    print("Proposals:", len(dataset_df), "| per spec:", dataset_df.groupby("spec_id").size().to_dict())
else:
    if not DATASET_MANIFEST_PATH.exists():
        raise FileNotFoundError("REBUILD_DATASET=False nhưng chưa có dataset_manifest.csv")
    dataset_df = pd.read_csv(DATASET_MANIFEST_PATH, encoding="utf-8-sig")
    for optional_column in ["reference_point", "reference_identity"]:
        if optional_column not in dataset_df.columns:
            dataset_df[optional_column] = None
    required_proposal_columns = {
        "proposal_id", "background_id", "proposal_feasibility_score",
        "target_mask_path", "reference_mask_path", "expected_person_box",
        "feasibility_components",
    }
    missing = sorted(required_proposal_columns - set(dataset_df.columns))
    if missing:
        raise RuntimeError(
            f"Dataset cache chưa có SpatialProposal fields {missing}; "
            "hãy đặt REBUILD_DATASET=True."
        )
    if not PROPOSAL_MANIFEST_PATH.exists():
        dataset_df[PROPOSAL_MANIFEST_COLUMNS].to_csv(
            PROPOSAL_MANIFEST_PATH, index=False, encoding="utf-8-sig"
        )

display(dataset_df.head())


### 10.2. Tạo generation candidates

Mỗi dòng proposal manifest được khôi phục thành `SpatialProposal`, sau đó tạo pool `CandidateState` theo ba search round. Online protocol chỉ ghi state được cấp budget; full-fixed protocol ghi đủ 16 seed để làm oracle pool. Vì vòng lặp đi qua proposal thay vì background, một background có thể có nhiều target mask mà không gây nhập nhằng.

In [ ]:
def optional_manifest_value(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    if isinstance(value, str) and not value.strip():
        return None
    return value


def decode_manifest_json(value, default=None):
    value = optional_manifest_value(value)
    if value is None:
        return default
    return json.loads(value) if isinstance(value, str) else value


def spatial_proposal_from_manifest_row(row):
    row = row.to_dict() if hasattr(row, "to_dict") else dict(row)
    background_path = INPUT_DIR / row["background_path"]
    with Image.open(background_path) as background:
        image_width, image_height = background.size

    def coordinates(name):
        value = decode_manifest_json(row.get(name))
        return tuple(int(v) for v in value) if value is not None else None

    metadata = {
        "image_width": image_width,
        "image_height": image_height,
        "heap_rank": int(row["heap_rank"]),
        "background_path": row["background_path"],
        "semantic_mask_path": row["semantic_mask_path"],
        "object_label_ids": row["object_label_ids"],
        "reference_scale_range": row["reference_scale_range"],
    }
    return SpatialProposal(
        proposal_id=row["proposal_id"],
        spec_id=row["spec_id"],
        background_id=row["background_id"],
        target_box=coordinates("target_box"),
        target_mask_path=row["target_mask_path"],
        reference_box=coordinates("reference_box"),
        reference_mask_path=optional_manifest_value(row.get("reference_mask_path")),
        reference_point=coordinates("reference_point"),
        reference_identity=optional_manifest_value(row.get("reference_identity")),
        expected_person_box=coordinates("expected_person_box"),
        expected_scale=optional_manifest_value(row.get("expected_scale")),
        ground_line=optional_manifest_value(row.get("ground_line")),
        feasibility_components=decode_manifest_json(
            row.get("feasibility_components"), {}
        ),
        proposal_feasibility_score=float(row["proposal_feasibility_score"]),
        metadata=metadata,
    )


def resized_generation_size(background_path):
    with Image.open(INPUT_DIR / background_path) as background:
        scale = min(
            CONFIG["width"] / background.width,
            CONFIG["height"] / background.height,
        )
        return tuple(max(8, int(d * scale) // 8 * 8) for d in background.size)


def build_generation_params(proposal, seed):
    width, height = resized_generation_size(proposal.metadata["background_path"])
    specification = get_context_spec(proposal.spec_id)
    return {
        "model_id": CONFIG["model_id"],
        "width": width,
        "height": height,
        "num_inference_steps": CONFIG["num_inference_steps"],
        "guidance_scale": CONFIG["guidance_scale"],
        "strength": CONFIG["strength"],
        "mask_blur_radius": CONFIG["mask_blur_radius"],
        "seed": int(seed),
        "negative_prompt": specification.negative_prompt,
        "torch_dtype": "float16",
    }


proposal_lookup = {
    proposal.proposal_id: proposal
    for proposal in (
        spatial_proposal_from_manifest_row(row)
        for _, row in dataset_df.iterrows()
    )
}
if len(proposal_lookup) != len(dataset_df):
    raise ValueError("proposal_id bị trùng trong dataset manifest")

generation_seeds = list(range(
    CONFIG["seed_start"],
    CONFIG["seed_start"] + CONFIG["max_seeds_per_proposal"],
))


def split_seed_rounds(seeds, round_counts):
    rounds, offset = [], 0
    for count in round_counts:
        rounds.append(seeds[offset:offset + count])
        offset += count
    return rounds


generation_seed_rounds = split_seed_rounds(
    generation_seeds, CONFIG["adaptive_round_seed_counts"]
)


def build_candidate_state(proposal, seed, search_round):
    candidate_id = build_candidate_id(proposal.proposal_id, seed)
    output_path = (
        OUTPUT_DIR / "images" / proposal.spec_id / proposal.background_id
        / proposal.proposal_id / f"{candidate_id}.png"
    )
    return CandidateState(
        candidate_id=candidate_id, spec_id=proposal.spec_id,
        background_id=proposal.background_id, proposal_id=proposal.proposal_id,
        seed=seed, search_round=search_round,
        generation_params=build_generation_params(proposal, seed),
        output_path=str(output_path),
    )


candidate_pool = {
    (proposal.proposal_id, round_index): [
        build_candidate_state(proposal, seed, round_index)
        for seed in round_seeds
    ]
    for proposal in proposal_lookup.values()
    for round_index, round_seeds in enumerate(generation_seed_rounds)
}
all_candidate_states = [
    state for states in candidate_pool.values() for state in states
]
validate_candidate_states(all_candidate_states)
candidate_states = []  # Chỉ chứa candidate thực sự được cấp budget.
CANDIDATE_MANIFEST_COLUMNS = [
    "candidate_id", "spec_id", "context_type", "background_id",
    "proposal_id", "seed", "search_round", "parent_candidate_id",
    "output_path", "generation_params",
    "proposal_feasibility_score", "heap_rank",
]


def write_candidate_manifest(states):
    records = []
    for state in states:
        proposal = proposal_lookup[state.proposal_id]
        data = candidate_state_to_dict(
            state, proposal, get_context_spec(state.spec_id)
        )
        data["generation_params"] = stable_json(data["generation_params"])
        records.append({column: data.get(column) for column in CANDIDATE_MANIFEST_COLUMNS})
    pd.DataFrame(records).to_csv(
        CANDIDATE_MANIFEST_PATH, index=False, encoding="utf-8-sig"
    )


print(
    "Potential candidates:", len(all_candidate_states),
    "| proposals:", len(proposal_lookup),
    "| round sizes:", [len(seeds) for seeds in generation_seed_rounds],
)


## 11. Kiểm tra trực quan dataset cache (tùy chọn)

Khi `CREATE_PREVIEWS=True`, cell hiển thị background, semantic annotation và target mask cạnh nhau để kiểm tra vùng chân/ghế và reference object trước khi chạy SDXL. Cell được bỏ qua hoàn toàn ở cấu hình mặc định.


In [ ]:
if CREATE_PREVIEWS:
    for row in dataset_df.itertuples(index=False):
        background = Image.open(INPUT_DIR / row.background_path).convert("RGB")
        semantic = Image.open(INPUT_DIR / row.semantic_mask_path).convert("L")
        target = Image.open(INPUT_DIR / row.target_mask_path).convert("L")

        size = (300, 300)

        bg_thumb = background.copy()
        bg_thumb.thumbnail(size)

        sem_thumb = semantic.convert("RGB")
        sem_thumb.thumbnail(size)

        target_thumb = target.convert("RGB")
        target_thumb.thumbnail(size)

        canvas = Image.new("RGB", (900, 350), "white")
        canvas.paste(bg_thumb, (0, 0))
        canvas.paste(sem_thumb, (300, 0))
        canvas.paste(target_thumb, (600, 0))

        draw = ImageDraw.Draw(canvas)
        draw.text(
            (10, 315),
            f"{row.proposal_id} | {row.context_type} | "
            f"{row.reference_object}",
            fill="black",
        )

        display(canvas)
else:
    print("CREATE_PREVIEWS=False: bỏ qua dataset cache preview.")


## 12. Khởi tạo SDXL Inpainting

Generation yêu cầu GPU. Model dùng FP16 và CPU offload để giảm VRAM; negative prompt tập trung loại lỗi anatomy, nhiều người và biến dạng background.

In [ ]:
if RUN_GENERATION:
    assert torch.cuda.is_available(), "Generation cần GPU."
    torch.cuda.empty_cache()
    gc.collect()
    pipe = AutoPipelineForInpainting.from_pretrained(
        CONFIG["model_id"],
        torch_dtype=torch.float16,
        variant="fp16",
        use_safetensors=True,
    )
    pipe.enable_model_cpu_offload()
    pipe.set_progress_bar_config(disable=False)

    tokenizers = [t for t in (pipe.tokenizer, pipe.tokenizer_2) if t is not None]
    texts = [spec.prompt for spec in CONTEXT_SPECIFICATIONS] + [NEGATIVE_PROMPT]
    for text in texts:
        counts = [len(tokenizer(text, truncation=False).input_ids) for tokenizer in tokenizers]
        limits = [tokenizer.model_max_length for tokenizer in tokenizers]
        if any(count > limit for count, limit in zip(counts, limits)):
            raise ValueError(f"Prompt vượt CLIP token limit: counts={counts}, limits={limits}: {text}")
    print("Loaded:", CONFIG["model_id"])
else:
    print("RUN_GENERATION=False: bỏ qua diffusion model.")


## 13. Chuẩn bị input và sinh ảnh

Ảnh và mask được resize đồng tỷ lệ về bội số của 8. Mọi candidate dùng chung feathered mask của proposal, không crop/zoom riêng vùng mask. Bước composite cuối vẫn bị giới hạn bởi hard mask, nên pixel bên ngoài vùng cho phép được lấy nguyên vẹn từ background.

In [ ]:
def load_inpainting_inputs(state, proposal):
    """Load background và đúng target mask được định danh bởi proposal_id."""
    background = Image.open(
        INPUT_DIR / proposal.metadata["background_path"]
    ).convert("RGB")
    mask = Image.open(INPUT_DIR / proposal.target_mask_path).convert("L")
    size = (state.generation_params["width"], state.generation_params["height"])
    background = background.resize(size, Image.Resampling.LANCZOS)
    mask = mask.resize(size, Image.Resampling.NEAREST)
    blurred_mask = mask.filter(ImageFilter.GaussianBlur(
        state.generation_params["mask_blur_radius"]
    ))
    soft_mask = ImageChops.darker(mask, blurred_mask)
    return background, soft_mask

def generate_candidate(state, proposal, specification, background, soft_mask):
    """Sinh candidate từ state + proposal + specification đã liên kết."""
    if state.proposal_id != proposal.proposal_id:
        raise ValueError("CandidateState không khớp SpatialProposal")
    if state.spec_id != specification.spec_id:
        raise ValueError("CandidateState không khớp ContextSpecification")
    params = state.generation_params
    generator = torch.Generator(device="cuda").manual_seed(state.seed)
    generated = pipe(
        prompt=specification.prompt,
        negative_prompt=specification.negative_prompt,
        image=background, mask_image=soft_mask, generator=generator,
        num_inference_steps=params["num_inference_steps"],
        guidance_scale=params["guidance_scale"], strength=params["strength"],
        width=params["width"], height=params["height"],
    ).images[0]
    return Image.composite(generated, background, soft_mask)

## 14. Định nghĩa evaluator

Hard gate kiểm tra output hợp lệ, đúng một target person, placement tối thiểu, scale hợp lý và relation tối thiểu khi context yêu cầu. Pose/contact và preservation không loại candidate trực tiếp mà chỉ dùng để xếp hạng.

Mask2Former cung cấp person gate và person mask; detection confidence chỉ áp dụng ở ngưỡng detection và được lưu để audit. ViTPose đánh giá pose/contact. Reference và context preservation được đo trong phần edit mask không thuộc người. Anatomy/photorealism do reviewer chấm trong manual validation notebook riêng vì baseline không dùng CLIP hoặc một quality proxy chưa được kiểm định.


### 14.1. Constraint-level evaluation schema

Evaluator giữ riêng output validity, subject validity, satisfaction của từng constraint và failure profile liên tục. Với physical interaction, `interaction` là diagnostic constraint weight 0 và `weak_interaction` dùng `1 - contact_score`. Prompt alignment và quality chưa được đo nên giữ `None`, không bị diễn giải thành không có lỗi. Các ngưỡng `promising_soft_threshold`/`accept_soft_threshold` điều khiển budget allocation nhưng không tham gia hard gate, soft ranking hoặc Best-of-N success.

In [ ]:
FAILURE_EPSILON = 0.05


@dataclass
class ConstraintResult:
    name: str
    score: float | None
    applicable: bool
    importance: Literal["hard", "soft"]
    threshold: float
    weight: float
    passed: bool | None
    details: dict = field(default_factory=dict)

    def __post_init__(self):
        if self.importance not in {"hard", "soft"}:
            raise ValueError(f"importance không hợp lệ: {self.importance}")
        validate_unit_interval(self.threshold, f"{self.name}.threshold")
        if self.weight < 0:
            raise ValueError(f"{self.name}.weight không được âm")
        if not self.applicable:
            if self.score is not None or self.passed is not None:
                raise ValueError("Constraint không áp dụng phải có score=passed=None")
        elif self.score is not None:
            validate_unit_interval(self.score, f"{self.name}.score")
        if self.score is None and self.passed is not None:
            raise ValueError("Constraint chưa đo phải có passed=None")


@dataclass
class FailureProfile:
    output_failure: float = 0.0
    no_person: float = 0.0
    wrong_person_count: float = 0.0
    wrong_placement: float = 0.0
    wrong_scale: float = 0.0
    wrong_pose: float = 0.0
    weak_relation: float = 0.0
    weak_interaction: float = 0.0
    reference_damage: float = 0.0
    context_damage: float = 0.0
    low_prompt_alignment: float | None = None
    low_quality: float | None = None
    dominant_failure: str | None = None
    severity: float = 0.0

    def __post_init__(self):
        for name in FAILURE_COMPONENT_NAMES:
            value = getattr(self, name)
            if value is not None:
                validate_unit_interval(value, f"FailureProfile.{name}")
        validate_unit_interval(self.severity, "FailureProfile.severity")
        if self.dominant_failure is not None and self.dominant_failure not in FAILURE_COMPONENT_NAMES:
            raise ValueError(f"dominant_failure không hợp lệ: {self.dominant_failure}")
        if self.dominant_failure is None and self.severity != 0.0:
            raise ValueError("severity phải bằng 0 khi không có dominant_failure")


FAILURE_COMPONENT_NAMES = [
    "output_failure", "no_person", "wrong_person_count",
    "wrong_placement", "wrong_scale", "wrong_pose",
    "weak_relation", "weak_interaction", "reference_damage",
    "context_damage", "low_prompt_alignment", "low_quality",
]
ADAPTIVE_FAILURE_COMPONENTS = [
    "wrong_placement", "wrong_scale", "wrong_pose",
    "weak_relation", "weak_interaction",
    "reference_damage", "context_damage",
]


@dataclass
class EvaluationResult:
    candidate_id: str
    proposal_id: str
    spec_id: str
    background_id: str
    seed: int
    output_valid: bool
    subject_valid: bool
    hard_valid: bool
    constraint_results: list[ConstraintResult]
    soft_score: float
    failure_profile: FailureProfile
    failure_reasons: list[str]
    metadata: dict = field(default_factory=dict)

    def __post_init__(self):
        validate_unit_interval(self.soft_score, "soft_score")
        if self.subject_valid and not self.output_valid:
            raise ValueError("subject_valid không thể True khi output không hợp lệ")
        if self.hard_valid and not self.subject_valid:
            raise ValueError("hard_valid không thể True khi subject không hợp lệ")
        if not isinstance(self.failure_reasons, list):
            raise TypeError("failure_reasons phải là list[str]")


def constraint_result_to_dict(result):
    return {
        "name": result.name, "score": result.score,
        "applicable": result.applicable, "importance": result.importance,
        "threshold": result.threshold, "weight": result.weight,
        "passed": result.passed, "details": dict(result.details),
    }


def failure_profile_to_dict(profile):
    return {
        **{name: getattr(profile, name) for name in FAILURE_COMPONENT_NAMES},
        "dominant_failure": profile.dominant_failure,
        "severity": profile.severity,
    }


def make_constraint_results(specification, scores, details, output_valid):
    results = []
    for constraint in specification.constraints:
        applicable = bool(
            constraint.params.get("applicable", True)
            and not constraint.params.get("manual_only", False)
        )
        score = scores.get(constraint.name) if applicable else None
        if not output_valid and constraint.name != "output_valid":
            score = None
        passed = (
            score >= constraint.threshold
            if score is not None and constraint.importance == "hard"
            else None
        )
        results.append(ConstraintResult(
            name=constraint.name, score=score, applicable=applicable,
            importance=constraint.importance, threshold=constraint.threshold,
            weight=constraint.weight, passed=passed,
            details=dict(details.get(constraint.name, {})),
        ))
    return results


def hard_valid_from_constraints(results):
    active = [
        result for result in results
        if result.applicable and result.importance == "hard"
    ]
    return bool(active) and all(result.passed is True for result in active)


def make_failure_profile(
    output_valid, person_count, target_found, scores,
    relation_applicable, interaction_applicable, reference_applicable,
):
    unmeasured = {"low_prompt_alignment", "low_quality"}
    values = {
        name: (None if name in unmeasured else 0.0)
        for name in FAILURE_COMPONENT_NAMES
    }
    active = []

    def set_failure(name, value):
        values[name] = float(np.clip(value, 0.0, 1.0))
        active.append(name)

    if not output_valid:
        set_failure("output_failure", 1.0)
    else:
        set_failure("no_person", 0.0 if target_found else 1.0)
        count_error = 0.0
        if person_count > 1:
            count_error = (person_count - 1) / person_count
        set_failure("wrong_person_count", count_error)
        for failure_name, score_name in [
            ("wrong_placement", "placement"),
            ("wrong_scale", "scale"),
            ("wrong_pose", "pose_contact"),
            ("context_damage", "context_preservation"),
        ]:
            score = scores.get(score_name)
            if score is not None:
                set_failure(failure_name, 1.0 - score)
        if relation_applicable and scores.get("relation") is not None:
            set_failure("weak_relation", 1.0 - scores["relation"])
        if interaction_applicable and scores.get("interaction") is not None:
            set_failure("weak_interaction", 1.0 - scores["interaction"])
        if reference_applicable and scores.get("reference_preservation") is not None:
            set_failure("reference_damage", 1.0 - scores["reference_preservation"])

    ranked = [(name, values[name]) for name in active if values[name] > FAILURE_EPSILON]
    dominant_failure, severity = max(ranked, key=lambda item: item[1]) if ranked else (None, 0.0)
    return FailureProfile(
        **{name: values[name] for name in FAILURE_COMPONENT_NAMES},
        dominant_failure=dominant_failure, severity=severity,
    )


def adaptive_readiness(output_valid, subject_valid, hard_valid, soft_score):
    if not output_valid:
        search_status = "invalid"
    elif hard_valid:
        search_status = "accepted"
    elif subject_valid and soft_score >= CONFIG["promising_soft_threshold"]:
        search_status = "promising"
    else:
        search_status = "rejected"
    return {
        "search_status": search_status,
        "expansion_eligible": bool(
            output_valid and subject_valid and not hard_valid
            and search_status == "promising"
        ),
        "stopping_eligible": bool(
            hard_valid and soft_score >= CONFIG["accept_soft_threshold"]
        ),
    }


In [ ]:
if RUN_EVALUATION:
    assert torch.cuda.is_available(), "Evaluation cần GPU."
    torch.cuda.empty_cache()
    gc.collect()

    DEVICE = "cuda"
    instance_processor = AutoImageProcessor.from_pretrained(
        CONFIG["instance_segmentation_model_id"]
    )
    instance_model = Mask2FormerForUniversalSegmentation.from_pretrained(
        CONFIG["instance_segmentation_model_id"]
    ).to(DEVICE).eval()

    pose_processor = AutoImageProcessor.from_pretrained(CONFIG["pose_model_id"])
    pose_model = VitPoseForPoseEstimation.from_pretrained(
        CONFIG["pose_model_id"]
    ).to(DEVICE).eval()

    print("Evaluator models loaded.")


### 14.2. Geometry, masks và scale prior

Scale score kết hợp target-region fit với khoảng tỷ lệ người/reference object theo loại vật thể. Đây là prior mềm theo phối cảnh cục bộ, tốt hơn chỉ so người với mask nhưng vẫn cần manual validation.


In [ ]:
def resize_box(box, old_size, new_size):
    if box is None:
        return None
    sx, sy = new_size[0] / old_size[0], new_size[1] / old_size[1]
    x1, y1, x2, y2 = box
    return [x1 * sx, y1 * sy, x2 * sx, y2 * sy]


def box_area(box):
    return max(0.0, box[2] - box[0]) * max(0.0, box[3] - box[1])


def box_iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    intersection = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    return intersection / max(1e-6, box_area(a) + box_area(b) - intersection)


def center_score(person_box, target_box):
    px = (person_box[0] + person_box[2]) / 2
    py = (person_box[1] + person_box[3]) / 2
    tx = (target_box[0] + target_box[2]) / 2
    ty = (target_box[1] + target_box[3]) / 2
    diagonal = math.hypot(target_box[2] - target_box[0], target_box[3] - target_box[1])
    return float(np.exp(-2.0 * math.hypot(px - tx, py - ty) / max(1.0, diagonal)))


def placement_score(person_box, target_box):
    score = (
        0.55 * center_score(person_box, target_box)
        + 0.45 * box_iou(person_box, target_box)
    )
    return float(np.clip(score, 0, 1))


def interval_score(value, bounds, sigma=0.40):
    if bounds is None:
        return np.nan
    low, high = bounds
    if low <= value <= high:
        return 1.0
    boundary = low if value < low else high
    return float(np.exp(-abs(math.log(max(value, 1e-6) / boundary)) / sigma))


def scale_score(person_box, target_box, reference_box, reference_range):
    person_h = max(1.0, person_box[3] - person_box[1])
    target_h = max(1.0, target_box[3] - target_box[1])
    target_fit = float(np.exp(-abs(math.log(person_h / target_h)) / 0.45))
    if reference_box is None or reference_range is None:
        return target_fit, np.nan
    reference_h = max(1.0, reference_box[3] - reference_box[1])
    reference_fit = interval_score(person_h / reference_h, reference_range)
    return 0.40 * target_fit + 0.60 * reference_fit, reference_fit


def preservation_score(original, generated, region, scale=0.08):
    if int(region.sum()) < 32:
        return np.nan
    a = np.asarray(original, dtype=np.float32) / 255.0
    b = np.asarray(generated, dtype=np.float32) / 255.0
    mae = float(np.abs(a - b)[region].mean())
    return float(np.exp(-mae / scale))


### 14.3. Person mask và pose

Mask2Former thực hiện instance segmentation; person instance gần tâm target box nhất được chọn. ViTPose chạy top-down trên person box để đánh giá keypoint coverage, tư thế và contact. Với physical interaction, contact thuộc `pose_score`; `relation_score` dùng quan hệ hình học giữa person box và reference box để tránh tính contact hai lần.


In [ ]:
@torch.inference_mode()
def segment_people(image):
    inputs = instance_processor(images=image, return_tensors="pt").to(DEVICE)
    outputs = instance_model(**inputs)
    result = instance_processor.post_process_instance_segmentation(
        outputs,
        threshold=CONFIG["person_detection_threshold"],
        target_sizes=[(image.height, image.width)],
        return_binary_maps=True,
    )[0]
    maps = result.get("segmentation")
    if maps is None:
        return []
    maps = maps.detach().cpu().numpy()
    if maps.ndim == 2:
        maps = maps[None]
    people = []
    for index, info in enumerate(result["segments_info"]):
        label = instance_model.config.id2label.get(info["label_id"], "").lower()
        if label != "person" or index >= len(maps):
            continue
        mask = maps[index].astype(bool)
        box = bbox_from_mask(mask)
        if box is not None:
            people.append({"score": float(info["score"]), "mask": mask, "box": box})
    return people


def select_target_person(people, target_box):
    if not people:
        return None
    return max(people, key=lambda person: center_score(person["box"], target_box))


@torch.inference_mode()
def estimate_pose(image, person_box):
    x1, y1, x2, y2 = person_box
    boxes = np.asarray([[x1, y1, x2 - x1, y2 - y1]], dtype=np.float32)
    inputs = pose_processor(images=image, boxes=[boxes], return_tensors="pt").to(DEVICE)
    outputs = pose_model(**inputs)
    results = pose_processor.post_process_pose_estimation(
        outputs,
        boxes=[boxes],
        target_sizes=[(image.height, image.width)],
    )
    if not results or not results[0]:
        return None, None
    pose = results[0][0]
    return pose["keypoints"].detach().cpu().numpy(), pose["scores"].detach().cpu().numpy()


def angle_degrees(a, b, c):
    u, v = np.asarray(a) - np.asarray(b), np.asarray(c) - np.asarray(b)
    denominator = np.linalg.norm(u) * np.linalg.norm(v)
    if denominator < 1e-6:
        return np.nan
    cosine = np.clip(np.dot(u, v) / denominator, -1.0, 1.0)
    return float(np.degrees(np.arccos(cosine)))


def pose_metrics(keypoints, scores, pose_params, reference_box, target_box):
    if keypoints is None:
        return 0.0, 0.0, 0.0
    visible = scores >= CONFIG["keypoint_threshold"]
    coverage = float(visible.mean())
    leg_angles = []
    for hip, knee, ankle in [(11, 13, 15), (12, 14, 16)]:
        if visible[[hip, knee, ankle]].all():
            leg_angles.append(angle_degrees(keypoints[hip], keypoints[knee], keypoints[ankle]))

    pose_mode = pose_params.get("pose_mode", "standing")
    contact_mode = pose_params.get("contact_mode", "ground")
    if pose_mode == "sitting":
        angle_score = np.mean([np.exp(-((a - 100.0) / 40.0) ** 2) for a in leg_angles]) if leg_angles else 0.0
        hip_ids = [i for i in (11, 12) if visible[i]]
        if contact_mode == "reference" and hip_ids and reference_box is not None:
            hip = keypoints[hip_ids].mean(axis=0)
            rx1, ry1, rx2, ry2 = reference_box
            dx = max(rx1 - hip[0], 0, hip[0] - rx2)
            dy = max(ry1 - hip[1], 0, hip[1] - ry2)
            norm = max(1.0, target_box[3] - target_box[1])
            contact = float(np.exp(-3.0 * math.hypot(dx, dy) / norm))
        else:
            contact = 0.0
        pose = 0.45 * coverage + 0.35 * float(angle_score) + 0.20 * contact
    elif pose_mode == "standing":
        angle_score = np.mean([np.exp(-((a - 170.0) / 30.0) ** 2) for a in leg_angles]) if leg_angles else 0.0
        ankle_ids = [i for i in (15, 16) if visible[i]]
        if ankle_ids:
            ankle_y = float(keypoints[ankle_ids, 1].mean())
            norm = max(1.0, target_box[3] - target_box[1])
            contact = float(np.exp(-abs(ankle_y - target_box[3]) / (0.20 * norm)))
        else:
            contact = 0.0
        pose = 0.45 * coverage + 0.35 * float(angle_score) + 0.20 * contact
    else:
        raise ValueError(f"pose mode không được hỗ trợ: {pose_mode}")
    return float(np.clip(pose, 0, 1)), coverage, contact



### 14.4. Hard constraints và soft ranking

Hard validity trả lời “candidate có đạt các điều kiện tối thiểu không?” và gồm output valid, single target person, placement, scale và relation khi cần. Các constraint placement/scale/relation đồng thời tham gia ranking qua weight tương ứng nhưng chỉ xuất hiện một lần trong schema. Manual anatomy/photorealism không được trộn vào automatic score.

Mỗi metric được đóng gói đúng một lần thành `ConstraintResult`; constraint hard vẫn có thể tham gia ranking khi `weight > 0`. Constraint không áp dụng dùng `score=None`, `passed=None`. `soft_score` là scalar ranking duy nhất, còn `FailureProfile` giữ riêng từng hướng lỗi. `failure_reasons` dùng JSON list làm representation duy nhất.


In [ ]:
def ranking_weights(specification):
    """Lấy mọi constraint weight dương, độc lập với hard-gate role."""
    return {
        constraint.name: constraint.weight
        for constraint in specification.constraints
        if constraint.weight > 0
    }


def relation_score(person_box, reference_box, relation_params):
    relation_mode = relation_params.get("mode")
    if relation_mode is None:
        return np.nan
    if reference_box is None:
        return 0.0
    if relation_mode == "overlap":
        overlap_x = max(0.0, min(person_box[2], reference_box[2]) - max(person_box[0], reference_box[0]))
        min_width = max(1.0, min(person_box[2] - person_box[0], reference_box[2] - reference_box[0]))
        horizontal_overlap = overlap_x / min_width
        score = (
            0.65 * horizontal_overlap
            + 0.35 * min(1.0, 5.0 * box_iou(person_box, reference_box))
        )
        return float(np.clip(score, 0, 1))
    if relation_mode != "proximity":
        raise ValueError(f"relation mode không được hỗ trợ: {relation_mode}")
    pc = (person_box[0] + person_box[2]) / 2
    rc = (reference_box[0] + reference_box[2]) / 2
    scale = max(1.0, reference_box[2] - reference_box[0])
    distance = abs(pc - rc) / scale
    separation = 1.0 - min(1.0, box_iou(person_box, reference_box) * 3.0)
    return float(np.clip(0.65 * np.exp(-distance) + 0.35 * separation, 0, 1))


def weighted_soft_score(values, weights):
    available = {k: v for k, v in values.items() if k in weights and np.isfinite(v)}
    denominator = sum(weights[k] for k in available)
    if denominator == 0:
        return 0.0
    return float(sum(weights[k] * v for k, v in available.items()) / denominator)


def finite_or_none(value):
    return float(value) if value is not None and np.isfinite(value) else None


def evaluation_result_to_dict(result):
    profile = failure_profile_to_dict(result.failure_profile)
    return {
        **result.metadata,
        "candidate_id": result.candidate_id,
        "proposal_id": result.proposal_id,
        "spec_id": result.spec_id,
        "background_id": result.background_id,
        "seed": result.seed,
        "output_valid": int(result.output_valid),
        "subject_valid": int(result.subject_valid),
        "hard_valid": int(result.hard_valid),
        "constraint_results": stable_json([
            constraint_result_to_dict(item) for item in result.constraint_results
        ]),
        "soft_score": result.soft_score,
        "failure_profile": stable_json(profile),
        "failure_reasons": stable_json(result.failure_reasons),
        "dominant_failure": result.failure_profile.dominant_failure,
        "failure_severity": result.failure_profile.severity,
        **{name: profile[name] for name in FAILURE_COMPONENT_NAMES},
    }


def make_evaluation_result(run, specification, scores, details, audit, failure_reasons):
    output_valid = bool(scores["output_valid"])
    subject_valid = bool(scores.get("single_target_person", 0.0))
    constraints = make_constraint_results(
        specification, scores, details, output_valid
    )
    hard_valid = hard_valid_from_constraints(constraints)
    soft_score = float(audit["soft_score"])
    relation_spec = get_constraint_spec(specification, "relation", "hard")
    interaction_spec = get_constraint_spec(
        specification, "interaction", "soft"
    )
    reference_spec = get_constraint_spec(
        specification, "reference_preservation", "soft"
    )
    profile = make_failure_profile(
        output_valid=output_valid,
        person_count=int(audit["person_count"]),
        target_found=bool(audit["target_found"]),
        scores=scores,
        relation_applicable=relation_spec is not None,
        interaction_applicable=interaction_spec is not None,
        reference_applicable=reference_spec is not None,
    )
    readiness = adaptive_readiness(
        output_valid, subject_valid, hard_valid, soft_score
    )
    metadata = {
        "context_type": run.context_type,
        "heap_rank": run.heap_rank,
        "proposal_feasibility_score": run.proposal_feasibility_score,
        "search_round": int(run.search_round),
        "parent_candidate_id": (
            None if pd.isna(run.parent_candidate_id) else run.parent_candidate_id
        ),
        "output_path": run.output_path,
        **readiness,
        **{key: value for key, value in audit.items() if key != "target_found"},
    }
    return EvaluationResult(
        candidate_id=run.candidate_id,
        proposal_id=run.proposal_id,
        spec_id=run.spec_id,
        background_id=run.background_id,
        seed=int(run.seed),
        output_valid=output_valid,
        subject_valid=subject_valid,
        hard_valid=hard_valid,
        constraint_results=constraints,
        soft_score=soft_score,
        failure_profile=profile,
        failure_reasons=list(failure_reasons),
        metadata=metadata,
    )


def constraint_long_rows(results):
    rows = []
    for result in results:
        context_type = result.metadata["context_type"]
        for constraint in result.constraint_results:
            row = constraint_result_to_dict(constraint)
            row["details"] = stable_json(row["details"])
            rows.append({
                "candidate_id": result.candidate_id,
                "proposal_id": result.proposal_id,
                "spec_id": result.spec_id,
                "context_type": context_type,
                "background_id": result.background_id,
                "seed": result.seed,
                "search_round": result.metadata["search_round"],
                **row,
            })
    return rows


def failure_profile_rows(results):
    rows = []
    for result in results:
        rows.append({
            "candidate_id": result.candidate_id,
            "proposal_id": result.proposal_id,
            "spec_id": result.spec_id,
            "context_type": result.metadata["context_type"],
            "background_id": result.background_id,
            "seed": result.seed,
            "search_round": result.metadata["search_round"],
            "parent_candidate_id": result.metadata["parent_candidate_id"],
            "output_valid": int(result.output_valid),
            "subject_valid": int(result.subject_valid),
            "hard_valid": int(result.hard_valid),
            "search_status": result.metadata["search_status"],
            "expansion_eligible": result.metadata["expansion_eligible"],
            "stopping_eligible": result.metadata["stopping_eligible"],
            **failure_profile_to_dict(result.failure_profile),
        })
    return rows


def output_failure_result(run, specification, reason, error=""):
    scores = {"output_valid": 0.0}
    details = {
        "output_valid": {
            "status": run.status, "reason": reason, "error": error,
        }
    }
    audit = {
        "target_found": False,
        "person_count": 0,
        "detection_confidence": 0.0,
        "person_box": "null",
        "placement_score": 0.0,
        "scale_score": 0.0,
        "reference_scale_score": np.nan,
        "pose_score": 0.0,
        "pose_coverage": 0.0,
        "contact_score": 0.0,
        "relation_score": np.nan,
        "reference_preservation_score": np.nan,
        "context_preservation_score": np.nan,
        "soft_score": 0.0,
    }
    return make_evaluation_result(
        run, specification, scores, details, audit, [reason]
    )


### 14.5. Evaluate candidates

Cell này đọc output image và proposal metadata, chạy detector/pose estimator, rồi tạo `EvaluationResult` cho từng candidate.


In [ ]:
manifest_lookup = dataset_df.set_index("proposal_id").to_dict("index")

def evaluate_run_frame(run_frame):
    evaluation_results = []
    for run in run_frame.itertuples(index=False):
        specification = get_context_spec(run.spec_id)
        if run.status != "success":
            evaluation_results.append(output_failure_result(
                run, specification, "output_failed",
                getattr(run, "error", ""),
            ))
            continue

        try:
            image = Image.open(run.output_path).convert("RGB")
            image.load()
        except Exception as exc:
            evaluation_results.append(output_failure_result(
                run, specification, "invalid_output", repr(exc)
            ))
            continue

        meta = manifest_lookup[run.proposal_id]
        original_native = Image.open(INPUT_DIR / meta["background_path"]).convert("RGB")
        old_size = original_native.size
        original = original_native.resize(image.size, Image.Resampling.LANCZOS)
        hard_mask = np.asarray(
            Image.open(INPUT_DIR / meta["target_mask_path"])
            .convert("L").resize(image.size, Image.Resampling.NEAREST)
        ) > 127
        semantic = np.asarray(
            Image.open(INPUT_DIR / meta["semantic_mask_path"])
            .convert("L").resize(image.size, Image.Resampling.NEAREST)
        )
        target_box = resize_box(decode_manifest_json(meta["target_box"]), old_size, image.size)
        reference_box = resize_box(decode_manifest_json(meta["reference_box"]), old_size, image.size)
        reference_range = decode_manifest_json(meta["reference_scale_range"])
        pose_constraint = get_constraint_spec(
            specification, "pose_contact", "soft"
        )
        relation_constraint = get_constraint_spec(
            specification, "relation", "hard"
        )
        reference_constraint = get_constraint_spec(
            specification, "reference_preservation", "soft"
        )
        relation_applicable = relation_constraint is not None
        reference_applicable = reference_constraint is not None

        people = segment_people(image)
        selected = select_target_person(people, target_box)
        person_count = len(people)
        if selected is None:
            person_box, person_mask, detection_confidence = None, np.zeros_like(hard_mask), 0.0
            placement = scale = reference_scale = pose = pose_coverage = contact = 0.0
            relation = 0.0 if relation_applicable else np.nan
        else:
            person_box = selected["box"]
            person_mask = selected["mask"]
            detection_confidence = selected["score"]
            placement = placement_score(person_box, target_box)
            scale, reference_scale = scale_score(
                person_box, target_box, reference_box, reference_range
            )
            keypoints, keypoint_scores = estimate_pose(image, person_box)
            pose, pose_coverage, contact = pose_metrics(
                keypoints, keypoint_scores, pose_constraint.params,
                reference_box, target_box
            )
            relation = (
                relation_score(person_box, reference_box, relation_constraint.params)
                if relation_applicable else np.nan
            )

        person_dilated = ndimage.binary_dilation(person_mask, iterations=5)
        object_ids = decode_manifest_json(meta["object_label_ids"], [])
        reference_mask = np.isin(semantic, object_ids) if object_ids else np.zeros_like(hard_mask)
        reference_visible = hard_mask & reference_mask & ~person_dilated
        reference_preservation = preservation_score(original, image, reference_visible)
        context_region = hard_mask & ~person_dilated & ~ndimage.binary_dilation(reference_mask, iterations=3)
        context_preservation = preservation_score(original, image, context_region)

        soft_values = {
            "placement": placement,
            "scale": scale,
            "pose_contact": pose,
            "relation": relation,
            "reference_preservation": reference_preservation,
            "context_preservation": context_preservation,
        }
        soft_score = weighted_soft_score(soft_values, ranking_weights(specification))
        scores = {
            "output_valid": 1.0,
            "single_target_person": float(selected is not None and person_count == 1),
            "placement": float(placement),
            "scale": float(scale),
            "pose_contact": float(pose),
            "relation": finite_or_none(relation) if relation_applicable else None,
            "reference_preservation": (
                finite_or_none(reference_preservation) if reference_applicable else None
            ),
            "context_preservation": finite_or_none(context_preservation),
            "manual_photorealism": None,
            "interaction": float(contact),
        }
        details = {
            "output_valid": {"status": run.status},
            "single_target_person": {
                "person_count": person_count,
                "target_found": selected is not None,
                "detection_confidence": detection_confidence,
            },
            "placement": {"target_box": target_box, "person_box": person_box},
            "scale": {"reference_scale_score": reference_scale},
            "pose_contact": {
                "pose_coverage": pose_coverage,
                "contact_score": contact,
            },
            "interaction": {"contact_score": contact},
            "relation": {
                "mode": (relation_constraint.params.get("mode")
                         if relation_constraint is not None else None)
            },
            "reference_preservation": {"reference_box": reference_box},
            "context_preservation": {},
            "manual_photorealism": {"measurement_available": False},
        }

        failure_reasons = []
        if selected is None:
            failure_reasons.append("no_person")
        else:
            if person_count != 1:
                failure_reasons.append("person_count")
            if placement < hard_threshold(specification, "placement"):
                failure_reasons.append("placement")
            if scale < hard_threshold(specification, "scale"):
                failure_reasons.append("scale")
            if (relation_applicable
                    and relation < hard_threshold(specification, "relation")):
                failure_reasons.append("relation")

        audit = {
            "target_found": selected is not None,
            "person_count": person_count,
            "detection_confidence": detection_confidence,
            "person_box": stable_json(person_box),
            "placement_score": placement,
            "scale_score": scale,
            "reference_scale_score": reference_scale,
            "pose_score": pose,
            "pose_coverage": pose_coverage,
            "contact_score": contact,
            "relation_score": relation,
            "reference_preservation_score": reference_preservation,
            "context_preservation_score": context_preservation,
            "soft_score": soft_score,
        }
        result = make_evaluation_result(
            run, specification, scores, details, audit, failure_reasons
        )
        if result.hard_valid != (len(failure_reasons) == 0):
            raise AssertionError("Constraint hard gate lệch logic baseline")
        evaluation_results.append(result)

    return evaluation_results


### 14.6. Evaluation outputs

Chuẩn hóa candidate audit, constraint long-form, failure profile và ranking CSV.


In [ ]:
def evaluation_frames(results):
    evaluation_df = pd.DataFrame([
        evaluation_result_to_dict(result) for result in results
    ])
    constraint_results_df = pd.DataFrame(constraint_long_rows(results))
    failure_profiles_df = pd.DataFrame(failure_profile_rows(results))
    return evaluation_df, constraint_results_df, failure_profiles_df


def write_evaluation_outputs(results):
    evaluation_df, constraint_results_df, failure_profiles_df = evaluation_frames(results)
    constraint_results_df.to_csv(
        CONSTRAINT_RESULTS_PATH, index=False, encoding="utf-8-sig"
    )
    failure_profiles_df.to_csv(
        FAILURE_PROFILES_PATH, index=False, encoding="utf-8-sig"
    )
    evaluation_df.to_csv(
        CANDIDATE_AUDIT_PATH, index=False, encoding="utf-8-sig"
    )
    ranking_group = ["spec_id", "background_id", "proposal_id"]
    ranking_df = evaluation_df.sort_values(
        ranking_group + ["hard_valid", "soft_score"],
        ascending=[True, True, True, False, False],
    )
    ranking_df["auto_rank"] = ranking_df.groupby(ranking_group).cumcount() + 1
    ranking_df.to_csv(OUTPUT_DIR / "ranking.csv", index=False, encoding="utf-8-sig")
    return evaluation_df, ranking_df, constraint_results_df, failure_profiles_df


def load_evaluation_outputs():
    ranking_path = OUTPUT_DIR / "ranking.csv"
    required_paths = [ranking_path, CONSTRAINT_RESULTS_PATH, FAILURE_PROFILES_PATH]
    missing_paths = [str(path) for path in required_paths if not path.exists()]
    if missing_paths:
        raise FileNotFoundError(f"Thiếu evaluator outputs: {missing_paths}")
    ranking_df = pd.read_csv(ranking_path, encoding="utf-8-sig")
    evaluation_df = ranking_df.drop(columns=["auto_rank"], errors="ignore")
    constraint_results_df = pd.read_csv(CONSTRAINT_RESULTS_PATH, encoding="utf-8-sig")
    failure_profiles_df = pd.read_csv(FAILURE_PROFILES_PATH, encoding="utf-8-sig")
    required_columns = {
        "candidate_id", "proposal_id", "background_id",
        "output_valid", "subject_valid",
        "failure_profile", "dominant_failure",
        "failure_severity", "search_status", "expansion_eligible",
        "stopping_eligible",
    }
    missing_columns = sorted(required_columns - set(evaluation_df.columns))
    if missing_columns:
        raise RuntimeError(
            f"ranking.csv chưa có standardized evaluator fields {missing_columns}; "
            "hãy đặt RUN_EVALUATION=True."
        )
    return evaluation_df, ranking_df, constraint_results_df, failure_profiles_df


## 15. Adaptive generation và evaluation theo vòng

Ngân sách tối đa là 16 seed/proposal theo ba tranche 4 → 4 → 8. `generation_protocol="full_fixed"` sinh đủ pool rồi mô phỏng policy offline để đo saving/regret chính xác; `"online_adaptive"` chỉ sinh các proposal được cấp tiếp budget. `search_round` là allocation metadata, không nằm trong candidate identity. Context damage chỉ được audit, không còn là hard early-stop. `all_no_person` chỉ mô tả các output hợp lệ đã quan sát; policy cần đủ `adaptive_min_no_person_seeds` output hợp lệ trước khi dừng vì lý do này. `RUN_GENERATION=False` cho phép đánh giá lại `runs.csv` hiện có.


In [ ]:
def generate_state_batch(states):
    records = []
    for state in states:
        proposal = proposal_lookup[state.proposal_id]
        specification = get_context_spec(state.spec_id)
        candidate_data = candidate_state_to_dict(state, proposal, specification)
        background, soft_mask = load_inpainting_inputs(state, proposal)
        output_path = Path(state.output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        record = {
            **candidate_data,
            "generation_params": stable_json(state.generation_params),
            "status": "started",
            "elapsed_seconds": None,
            "error": "",
        }
        try:
            start = time.time()
            result = generate_candidate(
                state, proposal, specification, background, soft_mask
            )
            result.save(output_path)
            state.output_path = str(output_path)
            record["output_path"] = state.output_path
            record["elapsed_seconds"] = time.time() - start
            record["status"] = "success"
            print(state.candidate_id, f'{record["elapsed_seconds"]:.1f}s')
        except Exception as exc:
            record["status"] = "failed"
            record["error"] = repr(exc)
            print("FAILED", state.candidate_id, exc)
        records.append(record)
    return records


def near_hard_threshold(row):
    margin = CONFIG["adaptive_near_hard_margin"]
    if row["output_valid"] != 1 or row["subject_valid"] != 1:
        return False
    specification = get_context_spec(row["spec_id"])
    checks = [
        row["placement_score"] >= hard_threshold(specification, "placement") - margin,
        row["scale_score"] >= hard_threshold(specification, "scale") - margin,
    ]
    if pd.notna(row["relation_score"]):
        checks.append(
            row["relation_score"]
            >= hard_threshold(specification, "relation") - margin
        )
    return all(checks)


def adaptive_policy_from_signals(signals, last_round):
    if last_round:
        return "stop", "budget_exhausted"
    if not CONFIG["adaptive_budget_enabled"]:
        return "continue", "adaptive_budget_disabled"
    if signals["num_valid_outputs"] == 0:
        return "stop", "all_outputs_invalid"
    if signals["stopping_eligible"]:
        return "stop", "high_quality_hard_valid"
    if signals["all_no_person"]:
        if (signals["no_person_evidence_count"]
                < CONFIG["adaptive_min_no_person_seeds"]):
            return "continue", "insufficient_no_person_evidence"
        return "stop", "all_no_person"
    if signals["dominant_failure"] == "wrong_pose" and signals["has_person"]:
        return "continue", "pose_seed_variation"
    if (signals["dominant_failure"] == "weak_relation"
            and signals["near_hard"]):
        return "continue", "relation_near_threshold"
    if (signals["dominant_failure"] == "wrong_scale"
            and not signals["near_hard"]):
        return "stop", "persistent_scale_mismatch"
    if signals["has_person"] and (signals["promising"] or signals["near_hard"]):
        reason = "promising" if signals["promising"] else "near_hard_threshold"
        return "continue", reason
    return "stop", "low_improvement_potential"


def adaptive_budget_decision(group, last_round):
    valid_output = group[group["output_valid"] == 1]
    has_hard_valid = bool(group["hard_valid"].eq(1).any())
    best_soft_score = float(group["soft_score"].max())
    no_person_observations = valid_output["no_person"].dropna()
    no_person_rate = (
        float(no_person_observations.mean())
        if len(no_person_observations) else np.nan
    )
    context_damage = valid_output["context_damage"].dropna()
    mean_context_damage = (
        float(context_damage.mean()) if len(context_damage) else np.nan
    )
    persistent_context_damage = bool(
        len(context_damage) == len(valid_output) and len(context_damage)
        and (context_damage >= CONFIG["adaptive_severe_context_damage"]).all()
    )
    subject_rows = group[group["subject_valid"] == 1]
    failure_means = {
        name: float(subject_rows[name].dropna().mean())
        if len(subject_rows[name].dropna()) else np.nan
        for name in ADAPTIVE_FAILURE_COMPONENTS
    }
    ranked_failures = [
        (name, value) for name, value in failure_means.items()
        if np.isfinite(value) and value > FAILURE_EPSILON
    ]
    dominant_failure, dominant_severity = (
        max(ranked_failures, key=lambda item: item[1])
        if ranked_failures else (None, 0.0)
    )

    signals = {
        "num_valid_outputs": len(valid_output),
        "stopping_eligible": bool(group["stopping_eligible"].astype(bool).any()),
        "all_no_person": bool(
            len(no_person_observations)
            and no_person_observations.eq(1.0).all()
        ),
        "no_person_evidence_count": len(no_person_observations),
        "has_person": bool(valid_output["person_count"].gt(0).any()),
        "near_hard": bool(group.apply(near_hard_threshold, axis=1).any()),
        "promising": best_soft_score >= CONFIG["promising_soft_threshold"],
        "dominant_failure": dominant_failure,
    }
    decision, reason = adaptive_policy_from_signals(signals, last_round)

    return {
        "decision": decision,
        "decision_reason": reason,
        "next_round_eligible": decision == "continue",
        "has_hard_valid": has_hard_valid,
        "best_soft_score": best_soft_score,
        "no_person_rate": no_person_rate,
        "no_person_evidence_count": len(no_person_observations),
        "mean_context_damage": mean_context_damage,
        "dominant_failure": dominant_failure,
        "dominant_failure_severity": dominant_severity,
        **{f"mean_{name}": value for name, value in failure_means.items()},
        "persistent_context_damage": persistent_context_damage,
    }


### 15.1. Chạy generation protocol

Thực thi full-fixed hoặc online-adaptive protocol, ghi run records và budget decisions sau mỗi round.


In [ ]:
runs_path = OUTPUT_DIR / "runs.csv"

if RUN_GENERATION:
    if not RUN_EVALUATION:
        raise ValueError("Generation theo vòng cần RUN_EVALUATION=True.")

    run_records = []
    evaluation_results = []
    budget_rows = []
    all_proposal_ids = set(proposal_lookup)
    policy_active_ids = set(all_proposal_ids)

    for round_index, _ in enumerate(generation_seed_rounds):
        generation_ids = (
            all_proposal_ids
            if CONFIG["generation_protocol"] == "full_fixed"
            else policy_active_ids
        )
        round_states = [
            state
            for proposal_id in sorted(generation_ids)
            for state in candidate_pool[(proposal_id, round_index)]
        ]
        if not round_states:
            break
        candidate_states.extend(round_states)
        round_records = generate_state_batch(round_states)
        run_records.extend(round_records)
        runs_df = pd.DataFrame(run_records)
        runs_df.to_csv(runs_path, index=False, encoding="utf-8-sig")
        write_candidate_manifest(candidate_states)

        if RUN_EVALUATION:
            evaluation_results.extend(evaluate_run_frame(pd.DataFrame(round_records)))
            evaluation_df, _, _ = evaluation_frames(evaluation_results)
            write_evaluation_outputs(evaluation_results)

        next_policy_active = set()
        last_round = round_index == len(generation_seed_rounds) - 1
        for proposal_id in sorted(policy_active_ids):
            group = evaluation_df[evaluation_df["proposal_id"] == proposal_id]
            policy = adaptive_budget_decision(group, last_round)
            budget_rows.append({
                "proposal_id": proposal_id,
                "spec_id": group.iloc[0]["spec_id"],
                "context_type": group.iloc[0]["context_type"],
                "background_id": group.iloc[0]["background_id"],
                "proposal_feasibility_score": group.iloc[0]["proposal_feasibility_score"],
                "round": round_index,
                "round_seed_count": len(generation_seed_rounds[round_index]),
                "cumulative_seed_count": len(group),
                "decision_source": CONFIG["generation_protocol"],
                **policy,
            })
            if policy["next_round_eligible"]:
                next_policy_active.add(proposal_id)

        pd.DataFrame(budget_rows).to_csv(
            BUDGET_DECISIONS_PATH, index=False, encoding="utf-8-sig"
        )
        print(
            f"Round {round_index}: generated={len(round_states)}, "
            f"policy_continue={len(next_policy_active)}"
        )
        policy_active_ids = next_policy_active
        if (CONFIG["generation_protocol"] == "online_adaptive"
                and not policy_active_ids):
            break

    validate_candidate_states(candidate_states)
    evaluation_df, ranking_df, constraint_results_df, failure_profiles_df = (
        write_evaluation_outputs(evaluation_results)
    )
else:
    if not runs_path.exists():
        raise FileNotFoundError("RUN_GENERATION=False nhưng chưa có runs.csv")
    runs_df = pd.read_csv(runs_path, encoding="utf-8-sig")
    if "search_round" not in runs_df:
        runs_df["search_round"] = 0
    if "parent_candidate_id" not in runs_df:
        runs_df["parent_candidate_id"] = None
    runs_df["candidate_id"] = [
        build_candidate_id(proposal_id, seed)
        for proposal_id, seed in zip(
            runs_df["proposal_id"], runs_df["seed"]
        )
    ]
    if runs_df["candidate_id"].duplicated().any():
        raise RuntimeError(
            "runs.csv chứa cùng proposal/seed nhiều lần; candidate identity bị trùng."
        )
    missing_parents = sorted(
        set(runs_df["parent_candidate_id"].dropna())
        - set(runs_df["candidate_id"])
    )
    if missing_parents:
        raise RuntimeError(f"runs.csv thiếu parent candidates: {missing_parents}")
    if RUN_EVALUATION:
        evaluation_results = evaluate_run_frame(runs_df)
        evaluation_df, ranking_df, constraint_results_df, failure_profiles_df = (
            write_evaluation_outputs(evaluation_results)
        )
    else:
        evaluation_df, ranking_df, constraint_results_df, failure_profiles_df = (
            load_evaluation_outputs()
        )

display(runs_df.groupby(["spec_id", "status"]).size().rename("count").reset_index())
display(ranking_df.head())


## 16. Tạo contact sheet (tùy chọn)

Khi `CREATE_PREVIEWS=True`, các output thành công được ghép theo proposal và ghi seed dưới mỗi ảnh. Cell được bỏ qua hoàn toàn ở cấu hình mặc định.


In [ ]:
def make_contact_sheet(group, proposal_id, cols=4, thumb=256):
    group = group[group["status"] == "success"]
    if group.empty:
        return None

    label_h = 34
    rows = math.ceil(len(group) / cols)

    sheet = Image.new("RGB", (cols * thumb, rows * (thumb + label_h)), "white")
    draw = ImageDraw.Draw(sheet)

    for index, row in enumerate(group.itertuples(index=False)):
        image = Image.open(row.output_path).convert("RGB")
        image.thumbnail((thumb, thumb))

        x = (index % cols) * thumb
        y = (index // cols) * (thumb + label_h)

        cell = Image.new("RGB", (thumb, thumb), "white")
        cell.paste(image, ((thumb - image.width) // 2, (thumb - image.height) // 2))
        sheet.paste(cell, (x, y))
        draw.text((x + 8, y + thumb + 8), f"seed={row.seed}", fill="black")

    path = OUTPUT_DIR / "contact_sheets" / f"{proposal_id}.png"
    sheet.save(path)
    return path


if CREATE_PREVIEWS:
    contact_sheet_paths = []
    for proposal_id, group in runs_df.groupby("proposal_id"):
        path = make_contact_sheet(group, proposal_id)
        if path:
            contact_sheet_paths.append(path)
    print(f"Saved {len(contact_sheet_paths)} contact sheets.")
    for path in contact_sheet_paths[:3]:
        display(Image.open(path))
else:
    print("CREATE_PREVIEWS=False: bỏ qua contact sheets.")


## 17. Failure summary

`summary_by_failure.csv` mô tả failure mode hậu nghiệm. Proposal-level outcome, variance và Best-of-N được hợp nhất ở phần kế tiếp để tránh hai bảng summary chồng lấn.

In [ ]:
failure_summary_source = evaluation_df.assign(
    dominant_failure_group=evaluation_df["dominant_failure"].fillna("none")
)
summary_by_failure = failure_summary_source.groupby(
    ["context_type", "spec_id", "dominant_failure_group"],
    dropna=False,
).agg(
    num_candidates=("candidate_id", "count"),
    mean_soft_score=("soft_score", "mean"),
    hard_valid_rate=("hard_valid", "mean"),
    mean_failure_severity=("failure_severity", "mean"),
).reset_index().rename(columns={
    "dominant_failure_group": "dominant_failure"
})
summary_by_failure["fraction"] = summary_by_failure["num_candidates"] / (
    summary_by_failure.groupby(["context_type", "spec_id"])["num_candidates"]
    .transform("sum")
)
summary_by_failure.to_csv(
    OUTPUT_DIR / "summary_by_failure.csv", index=False, encoding="utf-8-sig"
)


def dominant_mode(values):
    values = values.dropna()
    modes = values.mode()
    return modes.iloc[0] if len(modes) else None


display(summary_by_failure.head())



## 18. Within-proposal và between-proposal variance

`valid_candidate_score` được suy ra từ `hard_valid` và `soft_score`, không lưu lặp trong `ranking.csv`. Within-proposal variance phản ánh latent seed; between-proposal variance phản ánh chênh lệch mean giữa các spatial proposal trong cùng specification. Với online adaptive run, các estimate mang selection bias từ stopping rule; full-fixed protocol cung cấp fixed pool để so sánh công bằng.


In [ ]:
def build_budget_table(frame, budgets):
    rows = []
    for proposal_id, group in frame.groupby("proposal_id"):
        group = group.sort_values("seed")
        meta = group.iloc[0]
        for budget in budgets:
            prefix = group.head(budget)
            budget_reached = len(group) >= budget
            valid = prefix[prefix["hard_valid"] == 1]
            chosen_pool = valid if len(valid) else prefix
            chosen = chosen_pool.sort_values(
                ["soft_score", "seed"], ascending=[False, True]
            ).iloc[0]
            rows.append({
                "proposal_id": proposal_id,
                "spec_id": meta["spec_id"],
                "context_type": meta["context_type"],
                "background_id": meta["background_id"],
                "proposal_feasibility_score": meta["proposal_feasibility_score"],
                "budget": budget,
                "generation_protocol": CONFIG["generation_protocol"],
                "budget_reached": budget_reached,
                "num_observed_candidates": len(prefix),
                "success": int(len(valid) > 0) if budget_reached else np.nan,
                "has_valid_candidate": bool(len(valid) > 0),
                "num_valid": len(valid),
                "best_score": (
                    float(valid["soft_score"].max()) if len(valid)
                    else (0.0 if budget_reached else np.nan)
                ),
                "best_candidate_id": chosen["candidate_id"],
                "best_dominant_failure": chosen["dominant_failure"],
                "best_failure_severity": chosen["failure_severity"],
                "best_observed_soft_score": chosen["soft_score"],
            })
    return pd.DataFrame(rows)


analysis_df = evaluation_df.assign(
    valid_candidate_score=evaluation_df["soft_score"].where(
        evaluation_df["hard_valid"].eq(1), 0.0
    )
)
budget_df = build_budget_table(analysis_df, CONFIG["best_of_budgets"])
budget_features = budget_df.pivot(
    index="proposal_id", columns="budget", values="best_score"
).rename(columns=lambda budget: f"best_of_{budget}_score").reset_index()

proposal_stats = analysis_df.groupby(
    [
        "proposal_id", "spec_id", "context_type", "background_id",
        "heap_rank", "proposal_feasibility_score",
    ]
).agg(
    num_seeds=("seed", "count"),
    output_valid_rate=("output_valid", "mean"),
    subject_valid_rate=("subject_valid", "mean"),
    hard_valid_rate=("hard_valid", "mean"),
    mean_soft_score=("soft_score", "mean"),
    best_soft_score=("soft_score", "max"),
    mean_valid_candidate_score=("valid_candidate_score", "mean"),
    within_proposal_variance=("valid_candidate_score", "var"),
    mean_failure_severity=("failure_severity", "mean"),
    dominant_failure_mode=("dominant_failure", dominant_mode),
).reset_index().merge(budget_features, on="proposal_id")
proposal_stats["proposal_feasibility_percentile"] = proposal_stats.groupby("spec_id")[
    "proposal_feasibility_score"
].rank(method="average", pct=True)

variance_rows = []
for (spec_id, context_type), group in proposal_stats.groupby(["spec_id", "context_type"]):
    within = float(group["within_proposal_variance"].mean())
    raw_between = float(group["mean_valid_candidate_score"].var(ddof=1))
    mean_seed_count = float(group["num_seeds"].mean())
    between = max(0.0, raw_between - within / max(1.0, mean_seed_count))
    total = within + between
    variance_rows.append({
        "spec_id": spec_id,
        "context_type": context_type,
        "num_proposals": len(group),
        "num_backgrounds": group["background_id"].nunique(),
        "mean_num_seeds": mean_seed_count,
        "mean_within_proposal_variance": within,
        "raw_variance_of_proposal_means": raw_between,
        "estimated_between_proposal_variance": between,
        "between_variance_fraction": between / total if total > 0 else np.nan,
    })

variance_df = pd.DataFrame(variance_rows)
variance_df.to_csv(OUTPUT_DIR / "variance_decomposition.csv", index=False, encoding="utf-8-sig")
proposal_stats.to_csv(OUTPUT_DIR / "summary_by_proposal.csv", index=False, encoding="utf-8-sig")
context_stats = analysis_df.groupby("context_type").agg(
    num_proposals=("proposal_id", "nunique"),
    num_candidates=("seed", "count"),
    hard_valid_rate=("hard_valid", "mean"),
    mean_valid_candidate_score=("valid_candidate_score", "mean"),
    mean_soft_score=("soft_score", "mean"),
).reset_index()
context_stats.to_csv(OUTPUT_DIR / "summary_by_context.csv", index=False, encoding="utf-8-sig")
display(variance_df)
display(context_stats)


## 19. Proposal feasibility và generation outcomes

Notebook báo Pearson và Spearman giữa `proposal_feasibility_score` với automatic success rate, mean generation score và Best-of-1/4/8/16. Đây là phân tích liên hệ hậu nghiệm có thể chịu adaptive stopping bias, không diễn giải feasibility như một bộ dự đoán chất lượng ảnh sinh.


In [ ]:
def correlation_rows(frame, x, outcomes, source, group="all"):
    rows = []
    for outcome in outcomes:
        valid = frame[[x, outcome]].dropna()
        if len(valid) < 3 or valid[x].nunique() < 2 or valid[outcome].nunique() < 2:
            pearson = spearman = np.nan
            pearson_p = spearman_p = np.nan
        else:
            pearson, pearson_p = stats.pearsonr(valid[x], valid[outcome])
            spearman, spearman_p = stats.spearmanr(valid[x], valid[outcome])
        rows.append({
            "source": source, "group": group,
            "predictor": x, "outcome": outcome, "n": len(valid),
            "pearson_r": pearson, "pearson_p": pearson_p,
            "spearman_rho": spearman, "spearman_p": spearman_p,
        })
    return rows


correlation_outcomes = [
    "hard_valid_rate", "mean_valid_candidate_score",
    "best_of_1_score", "best_of_4_score", "best_of_8_score", "best_of_16_score",
]
automatic_correlation_rows = correlation_rows(
    proposal_stats, "proposal_feasibility_score", correlation_outcomes, "automatic"
)
automatic_correlation_rows.extend(correlation_rows(
    proposal_stats,
    "proposal_feasibility_percentile",
    correlation_outcomes,
    "automatic_within_spec_normalized",
))
for context_type, group in proposal_stats.groupby("context_type"):
    automatic_correlation_rows.extend(correlation_rows(
        group, "proposal_feasibility_score", correlation_outcomes,
        "automatic", group=context_type,
    ))
correlation_df = pd.DataFrame(automatic_correlation_rows)
correlation_df.to_csv(
    OUTPUT_DIR / "proposal_feasibility_correlations.csv",
    index=False, encoding="utf-8-sig",
)
display(correlation_df)


## 20. Best-of-1/4/8/16

Các budget dùng cùng thứ tự seed và là các prefix lồng nhau. Fixed Best-of-N chỉ được tính khi `budget_reached=True`; budget chưa đạt có `success` và `best_score` bằng NaN nhưng vẫn giữ best observed candidate để audit. Khi có đủ 16 seed/proposal, `adaptive_vs_oracle.csv` mô phỏng policy trên cùng seed order và báo generation saving, missed success, false early-stop và regret so với fixed Best-of-16.


In [ ]:
budget_summary = budget_df.groupby(["context_type", "budget"]).agg(
    num_proposals=("proposal_id", "nunique"),
    budget_reach_rate=("budget_reached", "mean"),
    success_rate=("success", "mean"),
    mean_best_score=("best_score", "mean"),
    std_best_score=("best_score", "std"),
).reset_index()
budget_df.to_csv(OUTPUT_DIR / "best_of_n_by_proposal.csv", index=False, encoding="utf-8-sig")
budget_summary.to_csv(OUTPUT_DIR / "best_of_n_summary.csv", index=False, encoding="utf-8-sig")


def best_valid_candidate(frame):
    valid = frame[frame["hard_valid"] == 1]
    if valid.empty:
        return False, 0.0, None
    best = valid.sort_values(
        ["soft_score", "seed"], ascending=[False, True]
    ).iloc[0]
    return True, float(best["soft_score"]), best["candidate_id"]


def simulate_adaptive_policy(full_frame):
    active = set(full_frame["proposal_id"].unique())
    rows = []
    cumulative_budget = 0
    for round_index, round_size in enumerate(CONFIG["adaptive_round_seed_counts"]):
        cumulative_budget += round_size
        next_active = set()
        last_round = round_index == len(CONFIG["adaptive_round_seed_counts"]) - 1
        for proposal_id in sorted(active):
            observed = (
                full_frame[full_frame["proposal_id"] == proposal_id]
                .sort_values("seed").head(cumulative_budget)
            )
            policy = adaptive_budget_decision(observed, last_round)
            meta = observed.iloc[0]
            rows.append({
                "proposal_id": proposal_id, "spec_id": meta["spec_id"],
                "context_type": meta["context_type"],
                "background_id": meta["background_id"],
                "proposal_feasibility_score": meta["proposal_feasibility_score"],
                "round": round_index,
                "round_seed_count": round_size,
                "cumulative_seed_count": cumulative_budget,
                "decision_source": "offline_full_fixed",
                **policy,
            })
            if policy["next_round_eligible"]:
                next_active.add(proposal_id)
        active = next_active
    return pd.DataFrame(rows)


max_budget = CONFIG["max_seeds_per_proposal"]
pool_sizes = analysis_df.groupby("proposal_id").size()
oracle_available = bool(len(pool_sizes) and pool_sizes.ge(max_budget).all())
oracle_columns = [
    "proposal_id", "spec_id", "context_type", "background_id",
    "adaptive_budget", "fixed_budget", "generations_saved",
    "adaptive_success", "oracle_success", "missed_success",
    "false_early_stop", "adaptive_best_score", "oracle_best_score",
    "score_regret", "adaptive_best_candidate_id",
    "oracle_best_candidate_id", "adaptive_stop_reason",
]
oracle_rows = []
if oracle_available:
    simulated_decisions = simulate_adaptive_policy(analysis_df)
    simulated_decisions.to_csv(
        BUDGET_DECISIONS_PATH, index=False, encoding="utf-8-sig"
    )
    final_decisions = (
        simulated_decisions.sort_values(["proposal_id", "round"])
        .groupby("proposal_id").tail(1).set_index("proposal_id")
    )
    for proposal_id, full_group in analysis_df.groupby("proposal_id"):
        full_group = full_group.sort_values("seed").head(max_budget)
        decision = final_decisions.loc[proposal_id]
        adaptive_budget = int(decision["cumulative_seed_count"])
        adaptive_group = full_group.head(adaptive_budget)
        adaptive_success, adaptive_score, adaptive_id = best_valid_candidate(adaptive_group)
        oracle_success, oracle_score, oracle_id = best_valid_candidate(full_group)
        meta = full_group.iloc[0]
        missed_success = bool(oracle_success and not adaptive_success)
        oracle_rows.append({
            "proposal_id": proposal_id, "spec_id": meta["spec_id"],
            "context_type": meta["context_type"],
            "background_id": meta["background_id"],
            "adaptive_budget": adaptive_budget, "fixed_budget": max_budget,
            "generations_saved": max_budget - adaptive_budget,
            "adaptive_success": adaptive_success, "oracle_success": oracle_success,
            "missed_success": missed_success,
            "false_early_stop": bool(adaptive_budget < max_budget and missed_success),
            "adaptive_best_score": adaptive_score,
            "oracle_best_score": oracle_score,
            "score_regret": oracle_score - adaptive_score,
            "adaptive_best_candidate_id": adaptive_id,
            "oracle_best_candidate_id": oracle_id,
            "adaptive_stop_reason": decision["decision_reason"],
        })
oracle_df = pd.DataFrame(oracle_rows, columns=oracle_columns)
oracle_df.to_csv(ORACLE_COMPARISON_PATH, index=False, encoding="utf-8-sig")
oracle_summary_rows = []
if oracle_available:
    groups = [("__all__", oracle_df), *oracle_df.groupby("context_type")]
    for context_type, group in groups:
        fixed_generations = int(group["fixed_budget"].sum())
        saved = int(group["generations_saved"].sum())
        oracle_summary_rows.append({
            "context_type": context_type,
            "num_proposals": len(group),
            "fixed_generations": fixed_generations,
            "adaptive_generations": int(group["adaptive_budget"].sum()),
            "generations_saved": saved,
            "generation_saving_rate": saved / fixed_generations,
            "adaptive_success_rate": float(group["adaptive_success"].mean()),
            "oracle_success_rate": float(group["oracle_success"].mean()),
            "missed_successes": int(group["missed_success"].sum()),
            "false_early_stop_rate": float(group["false_early_stop"].mean()),
            "mean_score_regret": float(group["score_regret"].mean()),
        })
oracle_summary_df = pd.DataFrame(oracle_summary_rows)
oracle_summary_df.to_csv(ORACLE_SUMMARY_PATH, index=False, encoding="utf-8-sig")


display(budget_summary)
if oracle_available:
    display(oracle_df.head())
    display(oracle_summary_df)
else:
    print("Oracle comparison unavailable: cần đủ 16 seed cho mọi proposal.")


## 21. Lưu experiment manifest

Manifest ghi lại thiết kế specification × background × spatial proposal × latent seed, đường dẫn proposal/candidate manifest, evaluator, hard thresholds và Best-of-N budgets để các lần chạy có thể được đối chiếu nhất quán.


In [ ]:
experiment_manifest = {
    **CONFIG,
    "method_name": (
        "ALTO v1: Context-aware evaluator with adaptive latent sampling "
        "under a fixed generation budget"
    ),
    "created_at": datetime.now().isoformat(),
    "design": (
        "specification x background x spatial_proposal x adaptive_seed_round"
    ),
    "num_spatial_proposals": int(dataset_df["proposal_id"].nunique()),
    "num_specifications": int(dataset_df["spec_id"].nunique()),
    "num_backgrounds": int(dataset_df["background_id"].nunique()),
    "num_seeds": len(generation_seeds),
    "num_generated_candidates": int(len(runs_df)),
    "dataset_manifest_path": str(DATASET_MANIFEST_PATH),
    "proposal_manifest_path": str(PROPOSAL_MANIFEST_PATH),
    "candidate_manifest_path": str(CANDIDATE_MANIFEST_PATH),
    "proposal_search_audit_path": (
        str(PROPOSAL_SEARCH_AUDIT_PATH) if PROPOSAL_SEARCH_AUDIT_PATH.exists() else None
    ),
    "candidate_audit_path": (
        str(CANDIDATE_AUDIT_PATH) if CANDIDATE_AUDIT_PATH.exists() else None
    ),
    "constraint_results_path": str(CONSTRAINT_RESULTS_PATH),
    "failure_profiles_path": str(FAILURE_PROFILES_PATH),
    "adaptive_budget_decisions_path": (
        str(BUDGET_DECISIONS_PATH) if BUDGET_DECISIONS_PATH.exists() else None
    ),
    "oracle_comparison_available": oracle_available,
    "oracle_comparison_path": str(ORACLE_COMPARISON_PATH),
    "oracle_summary_path": str(ORACLE_SUMMARY_PATH),
    "actual_seed_count_by_proposal": {
        str(key): int(value)
        for key, value in runs_df.groupby("proposal_id").size().to_dict().items()
    },
    "hard_constraints": [
        "output_valid", "single_target_person",
        "placement", "scale", "relation_when_required",
    ],
    "soft_components": sorted({
        constraint.name
        for specification in CONTEXT_SPECIFICATIONS
        for constraint in specification.constraints
        if constraint.weight > 0
    }),
    "manual_review_components": [
        "manual_anatomy", "manual_pose_contact",
        "manual_context_preservation", "manual_photorealism",
        "manual_accept",
    ],
}
manifest_path = OUTPUT_DIR / "experiment_manifest.json"
manifest_path.write_text(
    json.dumps(experiment_manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Saved:", manifest_path)
